## ⚠️ Nota de manutenção — Token do GitHub (disparo externo)

O agendamento deste job passou a ser disparado externamente via **cron-job.org**, que chama a API do GitHub (`workflow_dispatch`) em vez de depender do `schedule:` interno do GitHub Actions (que atrasava em horário de pico).

- **Token usado**: Personal Access Token (fine-grained), escopo restrito ao repositório `webscraping-livelo-esfera`, permissão `Actions: Read and write`.
- **Validade**: 06/09/2027 — renovar antes dessa data em https://github.com/settings/tokens, senão os disparos do cron-job.org param de funcionar.
- Configuração dos cronjobs (horários, headers, URL): painel do cron-job.org.

## 📋 Nota de manutenção — Horários de envio e resumo dos Jabares

### Horários fixos de disparo (via cron-job.org)

**Dias de semana** (7 envios, ~2-3h de intervalo):

| Envio | Horário (Brasília) |
|---|---|
| 1 | 07:38 |
| 2 | 09:38 |
| 3 | 12:38 |
| 4 | 14:38 |
| 5 | 16:38 |
| 6 | 18:38 |
| 7 | 20:38 |

**Fim de semana** (7 envios, ~1-2h de intervalo):

| Envio | Horário (Brasília) |
|---|---|
| 1 | 09:38 |
| 2 | 11:38 |
| 3 | 12:38 |
| 4 | 14:38 |
| 5 | 15:38 |
| 6 | 17:38 |
| 7 | 18:38 |

### O que é enviado em cada envio (todo dia)

1. Perfumaria e Cosméticos (Natura, Boticário, Eudora) + Suplementos / Fitness 💪 (2 min de espera antes)
2. Roupas e Calçados Esportivos (Centauro, NetShoes, New Balance, Nike, Adidas) + Café ☕ (2 min de espera antes)
3. Moda (Renner, Riachuelo, C&A, Hering, Dafiti) + Vinho 🍷 (2 min de espera antes)
4. Beleza e Cosméticos (Beleza na Web, Época Cosméticos, Sephora, Oceane) + Mercado 🛒 (2 min de espera antes)
5. Varejo (Kabum, Ponto, Casas Bahia, Extra, Magalu, Mercado Livre, FastShop)
6. Transferências Bonificadas
7. Jabares Noturnos + Ranking do dia (top 5 promoções) + Farmácia e Medicamentos 💊🩺 (1 min de espera antes)

### Jabares extras por dia da semana (além dos 7 envios de sempre)

| Dia | Envio | Jabar |
|---|---|---|
| Segunda | 3 | TopCashback |
| Segunda | 5 | Visto Americano (Hit The Change) |
| ~~Terça~~ | ~~7~~ | ~~Cartão Ruby Stell~~ *(desativado temporariamente — colidia com o Get Your Guide)* |
| Terça | 7 | Passeios Get Your Guide |
| Quarta | 2 | Guias de Viagem |
| Quarta | 3 | Banco Inter |
| Quinta | 3 | TopCashback |
| Quinta | 6 | Planilha de Controle Financeiro *(movido do envio 3 pra não colidir com o TopCashback)* |
| Sexta | 4 | Freetours Civitatis |
| Sexta | 7 | Passeios Get Your Guide |
| Sábado | 2 | Guias de Viagem |
| Sábado | 3 | Banco Inter |
| Domingo | 2 | Produtos que eu indico para viagens |
| Domingo | 3 | Planilha de Controle Financeiro |

A Captura Geral de Promoções (fila de aprovação) só varre novas fontes nos envios 1 e 4 — nos outros envios ela só processa/envia o que já está aprovado na fila.

### Resumo no privado (número pessoal)

A mensagem de resumo (📊 "Envio X de 7", promoções enviadas/capturadas/erros e o link da planilha) agora é enviada em **toda execução**, mesmo quando não teve nada relevante pra reportar — antes ela era pulada nesses casos, o que fazia parecer que só chegava "em alguns envios".

### Sugestão de aprovação por IA (aba Análise) + aba Recusados

Toda promoção nova capturada (Passo 3) agora também recebe uma **sugestão da IA** (Gemini), gravada nas colunas `sugestao_ia` (Aprovar/Recusar) e `motivo_ia` da aba Análise — baseada numa amostra recente do que já foi enviado (aba Envios) e do que já foi recusado (aba Recusados, nova).

**Importante: por enquanto é só sugestão.** A decisão de verdade continua sendo manual, na coluna `status` — a ideia é acompanhar se a IA acerta antes de deixar ela decidir sozinha no futuro.

A aba **Recusados** também passou a existir de verdade: quando você marca uma linha como "Recusado" na Análise, ela agora é arquivada lá (em vez de ficar esquecida na Análise pra sempre) — isso também alimenta os exemplos que a IA usa pra julgar as próximas promoções.

A extração de `expira_em` continua priorizando o regex (`extrair_data_validade`, gratuito); a IA só é usada como complemento quando o regex não encontra nenhuma data no texto.

### Sistema de Alertas Pessoais (nova seção, logo após a API do WhatsApp)

Alertas pontuais pro número pessoal (nunca pro grupo), separados do resumo diário — avisam na hora, não só no resumo do fim do envio. Função genérica `enviar_alerta_pessoal(titulo, mensagem)`, reaproveitável pra novos alertas no futuro. Os 3 primeiros:

1. **Centauro bonificada** (dentro do envio 2): quando a Centauro paga 6 pontos ou mais em qualquer clube de fidelidade.
2. **Transferência bonificada** (dentro do envio 6): toda vez que uma transferência bonificada nova é enviada pro grupo, também avisa no privado.
3. **Compra de milhas com desconto** (dentro da Captura Geral, Passo 3): quando a IA classifica uma promoção nova capturada como sendo desse tipo (campo `eh_compra_milhas_desconto`, calculado na mesma chamada de IA que já gera a sugestão de aprovação, sem custo extra).

### Histórico de promoções bonificadas (aba Ranking_historico)

Nova aba **Ranking_historico**, mesmas colunas da Ranking_dia (`data, categoria, parceiro, nome_clube, pontuacao_clube, pontuacao_ajustada, analise_pontuacao`) — mas **nunca é limpa**. Toda vez que uma promoção é registrada na Ranking_dia (envios 1 a 5), a mesma linha também vai pra cá, construindo um histórico completo dia após dia.

Corrigido também um bug: o emoji 🟣 do Shopping Livelo nunca aparecia de verdade (comparação de texto com maiúscula/minúscula trocada em `coletar_promocoes`).

### Novas categorias: Suplementos / Fitness, Café, Vinho e Mercado

Mesmo modelo da Farmácia (catálogo oficial Livelo/Esfera via `extrair_parceiros_livelo()`/`extrair_parceiros_esfera()`, filtrado por nome, com link do parceiro na mensagem) — só que essas rodam intercaladas com as 4 primeiras categorias do dia, cada uma com **2 minutos de espera** antes de disparar:

- **💪 Suplementos / Fitness** (envio 1, depois da Perfumaria): Max Titanium, Probiótica (Livelo+Esfera), Soldiers Nutrition, iHerb (Esfera)
- **☕ Café** (envio 2, depois de Roupas e Calçados): Coffee Mais/Coffee++ (Livelo+Esfera), Assinatura Coffee Mais, Café Orfeu (Livelo), Café L'or, Café Store, Dolce Gusto, Nespresso, Pilão (Esfera)
- **🍷 Vinho** (envio 3, depois da Moda): Mistral (Livelo+Esfera), Divvino (Livelo), Concha Y Toro, Evino, Freixenet, Grand Cru, Shop Vinho, World Wine (Esfera)
- **🛒 Mercado** (envio 4, depois de Beleza e Cosméticos): Carrefour Mercado, Carrefour Shopping, Sam's Club, Sam's Club - E-commerce, Supernosso (Livelo), Carrefour, La Pastina (Esfera)

O **Extra** foi mantido só na categoria Varejo (não entra em Mercado, pra não duplicar).

### Extração Livelo e Esfera (parceiros e pontuação) — DESATIVADA (extração em massa)

Nova célula no final do notebook, coleta o catálogo completo de parceiros e pontuação da Livelo (via `requests`, leve) e da Esfera (via Playwright, já testado e funcionando no GitHub Actions) e grava na aba **"Extração Livelo e Esfera"**.

**Está desativada de propósito** (`EXTRACAO_PARCEIROS_ATIVA = False`) — essa extração em massa (todos os parceiros) ainda não é usada em nenhum lugar. Antes de ativar: criar a aba na planilha real e trocar a flag para `True` (o `playwright` já foi adicionado ao `requirements.txt` e o workflow já instala o navegador, por causa da categoria de Farmácia abaixo).

### Nova categoria: Farmácia e Medicamentos (envio 7)

Usa as mesmas funções `extrair_parceiros_livelo()`/`extrair_parceiros_esfera()` de cima (por isso a célula fica no final do notebook, depois delas serem definidas), mas **ao vivo** — filtra só os parceiros de farmácia (Farmácias App, Drogaria São Paulo, Drogarias Pacheco, Drogal na Livelo; Drogal, Drogaria Araújo, Drogaria Venâncio, Extrafarma, Pague Menos na Esfera) e manda pro grupo, no formato das outras categorias (incluindo o link do parceiro).

Roda dentro do envio 7, com **1 minuto de espera** antes (`time.sleep(60)`) pra não brigar com o Jabar do dia que roda logo antes. Não usa comparemania.com.br — só o catálogo oficial de cada programa.

A célula fica fisicamente ANTES de "Registro do envio 7" e "Ranking do dia - Top 5" — por isso as promoções de Farmácia já entram no Ranking_dia a tempo de concorrer no **top 5 daquele mesmo dia** também.

⚠️ **Bug corrigido em 2026-09**: até a implementação das 4 categorias abaixo, essa célula na verdade quebrava silenciosamente (`NameError`) toda vez que rodava, porque `extrair_parceiros_livelo()`/`extrair_parceiros_esfera()` só eram definidas lá no fim do notebook — DEPOIS da Farmácia. As funções foram movidas pra uma célula bem no início do notebook (logo após os imports), então agora funciona de verdade.

## Dados padronizados

### Instalando Bibliotecas


In [ ]:
pip install pandas

In [ ]:
pip install datetime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.7/270.7 kB 14.4 MB/s eta 0:00:00


### Importando Bibliotecas


In [ ]:
import requests
import pandas as pd
import re
import os
import datetime
import time
from bs4 import BeautifulSoup
from zoneinfo import ZoneInfo
import textwrap

### Funções de extração de parceiros (Livelo e Esfera)

In [ ]:
# =========================================================================
# 🗂️ FUNÇÕES DE EXTRAÇÃO DE PARCEIROS (Livelo e Esfera)
# =========================================================================
# Ficam aqui, bem no início do notebook (logo após os imports), porque
# várias categorias mais abaixo (Farmácia, Suplementos, Café, Vinho,
# Mercado) precisam chamá-las — e o Jupyter executa célula por célula, de
# cima pra baixo, então elas têm que estar definidas ANTES de qualquer
# célula que as usa.

def extrair_parceiros_livelo(url="https://www.livelo.com.br/juntar-pontos/todos-os-parceiros"):
    """Extrai parceiro/pontuação/link direto do HTML (requests +
    BeautifulSoup, sem navegador — o site já manda os dados prontos no
    HTML, cada parceiro é um <a> com uma <img alt="Logo X">)."""
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    parceiros = []
    vistos = set()
    for link in soup.select('a[href*="/juntar-pontos/parceiros/"]'):
        href = link.get("href", "")
        m = re.search(r'/parceiros/([^/"]+)/([A-Z0-9]+)', href)
        if not m:
            continue
        slug = m.group(1)
        if slug in vistos:
            continue

        img = link.find("img", alt=re.compile(r'^Logo '))
        if not img:
            continue
        nome = img["alt"].replace("Logo ", "").strip()

        texto_card = link.get_text(" ", strip=True)
        # Remove "Eram X pontos" (valor ANTIGO, antes de uma promoção) pra
        # não confundir com os valores atuais.
        texto_limpo = re.sub(r'Eram\s+\d+(?:[.,]\d+)?\s*pontos?', '', texto_card)

        pontos_matches = list(re.finditer(r'(?:Até\s+)?(\d+(?:[.,]\d+)?)\s*pontos?', texto_limpo))
        if not pontos_matches:
            continue
        pos_clube = texto_limpo.find("Clube")

        pontuacao = None
        pontuacao_clube = None
        for pm in pontos_matches:
            valor = float(pm.group(1).replace(",", "."))
            if pos_clube != -1 and pm.start() > pos_clube:
                if pontuacao_clube is None:
                    pontuacao_clube = valor
            elif pontuacao is None:
                pontuacao = valor

        if pontuacao is None:
            continue

        url_absoluta = href if href.startswith("http") else f"https://www.livelo.com.br{href}"
        parceiros.append({
            "parceiro": nome,
            "pontuacao": pontuacao,
            "pontuacao_clube": pontuacao_clube,
            "moeda": None,
            "link_parceiro": url_absoluta,
        })
        vistos.add(slug)

    return parceiros


def extrair_parceiros_esfera(url="https://www.esfera.com.vc/junte-pontos/junte-pontos/esf02163"):
    """Extrai parceiro/pontuação/link via Playwright (navegador real) — o
    site só carrega os parceiros via JavaScript. Import do playwright fica
    DENTRO da função de propósito: só precisa estar instalado quando essa
    função é realmente chamada (Farmácia, Vinho e Mercado usam; caso
    alguma delas nunca rode num dia, não faz diferença, mas o pacote já
    está no requirements.txt e o workflow já instala o navegador)."""
    from playwright.sync_api import sync_playwright

    with sync_playwright() as p:
        navegador = p.chromium.launch(headless=True)
        pagina = navegador.new_page(user_agent=(
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
        ))
        pagina.goto(url, timeout=45000, wait_until="networkidle")
        pagina.wait_for_timeout(15000)  # dá tempo do JS carregar os parceiros

        links_brutos = pagina.eval_on_selector_all(
            'a[href*="/p/"]',
            """els => els
                .filter(a => a.textContent.includes('Ganhe'))
                .map(a => ({ href: a.href, texto: a.textContent.trim() }))
            """,
        )
        navegador.close()

    parceiros = []
    for item in links_brutos:
        texto = item["texto"]
        m = re.match(r"^(.*?)\s*Ganhe\s+(.*)$", texto)
        if not m:
            continue
        nome = m.group(1).strip()
        frase = m.group(2).strip()

        m2 = re.match(
            r"^(?:Até\s+)?(\d+(?:[.,]\d+)?)\s*pts?\s+(?:a cada|por)\s+(?:(\d+)\s+)?(real|reais|d[óo]lar|dolares|carro assinado)",
            frase, re.I
        )
        if not m2:
            continue

        pontos = float(m2.group(1).replace(",", "."))
        divisor = float(m2.group(2)) if m2.group(2) else 1.0
        unidade = m2.group(3).lower()

        if "carro" in unidade:
            pontuacao = pontos
            moeda = None
        else:
            pontuacao = round(pontos / divisor, 3)
            moeda = "dolar" if "lar" in unidade else "real"

        parceiros.append({
            "parceiro": nome,
            "pontuacao": pontuacao,
            "pontuacao_clube": None,
            "moeda": moeda,
            "link_parceiro": item["href"],
        })

    return parceiros


def registrar_extracao_parceiros(aba, parceiros, programa_fidelidade, data_extracao):
    """Grava a lista de parceiros extraída na aba, em lote (append_rows) —
    bem mais rápido que gravar linha por linha quando são centenas de
    parceiros de uma vez."""
    linhas = []
    for p in parceiros:
        linhas.append([
            data_extracao,
            programa_fidelidade,
            p["parceiro"],
            p["pontuacao"],
            p.get("pontuacao_clube") or "",
            p.get("moeda") or "",
            p.get("link_parceiro", ""),
        ])
    if linhas:
        aba.append_rows(linhas, value_input_option="USER_ENTERED")


def analisar_pontuacao_extracao(pontuacao):
    """Mesma escala de análise usada em coletar_promocoes() lá em cima -
    Livelo/Esfera não dividem por 1,3 (a divisão é só pra normalizar
    OUTROS clubes contra Livelo/Esfera; aqui já é o valor direto deles).
    Reaproveitada pelas categorias baseadas no catálogo Livelo/Esfera
    (Farmácia, Suplementos, Café, Vinho, Mercado)."""
    if pontuacao >= 10:
        return '⭐⭐⭐⭐⭐ Nível 5 (Excelente / Raro) 😏: Promoção rara, aproveite sem medo!'
    elif pontuacao >= 8:
        return '⭐⭐⭐⭐ Nível 4 (Muito Bom) 😎: Ótima Promoção, daqui pra cima já vale muito!'
    elif pontuacao >= 5:
        return '⭐⭐⭐ Nível 3 (Bom) 😉: Bom momento para potencializar compras planejadas, mas se puder aguardar, tem coisa melhor!'
    elif pontuacao >= 3:
        return '⭐⭐ Nível 2 (Mediano) 🧐: Dá pra usar se você realmente já iria comprar, mas é melhor aguardar algo melhor ;)'
    else:
        return '⭐ Nível 1 (Ruim) 😡: Sugiro aguardar algo melhor'


def formatar_pontuacao_extracao(valor):
    return str(int(valor)) if valor == int(valor) else str(round(valor, 2))


### Modo de Teste no Colab (upload das credenciais de teste)

In [ ]:
# =========================================
# 🧪 MODO DE TESTE NO COLAB
# =========================================
# Detecta automaticamente se está rodando no Google Colab. Se estiver,
# pede upload dos arquivos de credenciais de TESTE e os injeta em
# os.environ — assim todo o resto do notebook (que já lê tudo via
# os.environ[...]) funciona sem precisar alterar mais nenhuma célula.
#
# No GitHub Actions o import do google.colab falha (pacote não existe
# nesse ambiente), então esse bloco é pulado automaticamente e as
# variáveis de ambiente reais (secrets do GitHub) são usadas normalmente.
# Ou seja: não precisa lembrar de "reverter" nada antes de subir pro ar.

try:
    from google.colab import files
    RODANDO_NO_COLAB = True
except ImportError:
    RODANDO_NO_COLAB = False

# Coloque True quando precisar subir os arquivos de novo (ex: trocou uma
# chave/credencial e o arquivo antigo ainda está em cache na sessão do
# Colab). Depois de subir o que precisava, pode voltar pra False.
FORCAR_REUPLOAD = False

if RODANDO_NO_COLAB:
    arquivos_necessarios = [
        "id_grupo_teste.txt",
        "instance_id.txt",
        "api_key.txt",
        "sheet_id.txt",
        "SHEET_ID_CONTROLE_ENVIOS.txt",
        "gemini_key.txt",
        "url_scraping.txt",
        "url_scraping_two.txt",
        "credenciais.json",
        "numero_pessoal.txt",
    ]

    if FORCAR_REUPLOAD:
        faltando = arquivos_necessarios
    else:
        faltando = [a for a in arquivos_necessarios if not os.path.exists(a)]

    if faltando:
        print(f"📤 Selecione estes arquivos no seletor abaixo (pode escolher só os que mudaram): {faltando}")
        files.upload()

    def ler_arquivo_teste(nome_arquivo):
        with open(nome_arquivo, "r") as f:
            return f.read().strip()

    os.environ["ID_GRUPO_PESSOAL_TESTES"]  = ler_arquivo_teste("id_grupo_teste.txt")
    os.environ["INSTANCE_ID"]              = ler_arquivo_teste("instance_id.txt")
    os.environ["API_KEY"]                  = ler_arquivo_teste("api_key.txt")
    os.environ["SHEET_ID"]                 = ler_arquivo_teste("sheet_id.txt")
    os.environ["SHEET_ID_CONTROLE_ENVIOS"] = ler_arquivo_teste("SHEET_ID_CONTROLE_ENVIOS.txt")
    os.environ["GEMINI_API_KEY"]           = ler_arquivo_teste("gemini_key.txt")
    os.environ["URL_SCRAPING"]             = ler_arquivo_teste("url_scraping.txt")
    os.environ["URL_SCRAPING_TWO"]         = ler_arquivo_teste("url_scraping_two.txt")
    os.environ["GOOGLE_CREDENTIALS"]       = ler_arquivo_teste("credenciais.json")
    os.environ["NUMERO_PESSOAL"]           = ler_arquivo_teste("numero_pessoal.txt")

    print("✅ Variáveis de ambiente de teste carregadas a partir dos arquivos enviados no Colab.")
else:
    print("➡️ Não estamos no Colab — usando as variáveis de ambiente do GitHub Actions normalmente.")

📤 Selecione estes arquivos no seletor abaixo (pode escolher só os que mudaram): ['id_grupo_teste.txt', 'instance_id.txt', 'api_key.txt', 'sheet_id.txt', 'SHEET_ID_CONTROLE_ENVIOS.txt', 'gemini_key.txt', 'url_scraping.txt', 'url_scraping_two.txt', 'credenciais.json']


Saving gemini_key.txt to gemini_key.txt
Saving SHEET_ID_CONTROLE_ENVIOS.txt to SHEET_ID_CONTROLE_ENVIOS.txt
Saving api_key.txt to api_key.txt
Saving url_scraping.txt to url_scraping.txt
Saving credenciais.json to credenciais.json
Saving openai_key.txt to openai_key.txt
Saving id_grupo_teste.txt to id_grupo_teste.txt
Saving url_scraping_two.txt to url_scraping_two.txt
Saving instance_id.txt to instance_id.txt
Saving id_grupo.txt to id_grupo.txt
Saving sheet_id.txt to sheet_id.txt
✅ Variáveis de ambiente de teste carregadas a partir dos arquivos enviados no Colab.


### Definindo o ID do Grupo

In [ ]:
# 🚦 FLAG ÚNICA: para onde as mensagens são enviadas.
# True  -> grupo de TESTES (variável de ambiente ID_GRUPO_PESSOAL_TESTES)
# False -> grupo REAL (variável de ambiente ID_GRUPO_MILHAS)
#
# Essa é a única coisa que você precisa mudar para trocar entre testar e
# publicar de verdade. Antes de deixar o GitHub Actions rodar sozinho nos
# horários automáticos, confirme que está False.
ENVIAR_PARA_GRUPO_TESTE = False

if ENVIAR_PARA_GRUPO_TESTE:
    id_grupo_envio = os.environ['ID_GRUPO_PESSOAL_TESTES']
    print("🧪 ENVIAR_PARA_GRUPO_TESTE = True -> enviando para o GRUPO DE TESTES")
else:
    id_grupo_envio = os.environ['ID_GRUPO_MILHAS']
    print("🚀 ENVIAR_PARA_GRUPO_TESTE = False -> enviando para o GRUPO REAL")

🧪 ENVIAR_PARA_GRUPO_TESTE = True -> enviando para o GRUPO DE TESTES


### Definindo os dados da API para envio no Whatsapp

In [ ]:
INSTANCE_ID = os.environ['INSTANCE_ID']
API_KEY = os.environ['API_KEY']

# Endpoint
url = f"https://api.zapperapi.com/{INSTANCE_ID}/messages/text" #url para mensagens de texto
url_media = f"https://api.zapperapi.com/{INSTANCE_ID}/messages/media" #url para mensagens de texto com imagens

headers = {
    "X-Api-Key": API_KEY,
    "Content-Type": "application/json"
}

## 🔔 Sistema de Alertas Pessoais

Alertas pontuais mandados pro **número pessoal** (nunca pro grupo), separados do resumo diário — pra avisar na hora quando algo específico acontece, em vez de esperar o resumo do fim do envio. Novos alertas podem ser adicionados aqui no futuro.

In [ ]:
# ── NUMERO_PESSOAL: mesma lógica usada no resumo da Captura Geral, repetida
# aqui de propósito pra este sistema de alertas não depender da ordem de
# execução das células (funciona mesmo antes da célula de Captura Geral). ──
def obter_numero_pessoal_alerta():
    numero = os.environ.get("NUMERO_PESSOAL")
    if numero:
        return numero.strip()
    if os.path.exists("numero_pessoal.txt"):
        with open("numero_pessoal.txt", "r") as f:
            return f.read().strip()
    return None


def enviar_alerta_pessoal(titulo_alerta, mensagem):
    """Envia um alerta pro número PESSOAL (nunca pro grupo). Usado pelos
    alertas cadastrados nas células correspondentes (Centauro bonificada,
    Transferência bonificada, Compra de milhas com desconto, e outros que
    vierem no futuro).

    Define seus próprios headers (não reaproveita a variável global
    `headers`) porque algumas células do notebook reusam esse nome pra
    outra coisa (ex: headers de scraping) — assim este envio nunca quebra
    por causa disso.
    """
    numero = obter_numero_pessoal_alerta()
    if not numero:
        print(f"⚠️ NUMERO_PESSOAL não configurado — alerta '{titulo_alerta}' não enviado.")
        return False

    texto = f"🔔 *Alerta: {titulo_alerta}*\n\n{mensagem}"
    headers_alerta = {"X-Api-Key": API_KEY, "Content-Type": "application/json"}
    payload = {"jid": numero, "message": texto}
    try:
        resposta = requests.post(url, json=payload, headers=headers_alerta, timeout=30)
        if resposta.status_code == 200:
            print(f"🔔 Alerta '{titulo_alerta}' enviado pro número pessoal.")
            return True
        print(f"⚠️ Erro ao enviar alerta '{titulo_alerta}': {resposta.status_code} - {resposta.text}")
        return False
    except Exception as e:
        print(f"⚠️ Exceção ao enviar alerta '{titulo_alerta}': {e}")
        return False


### Função para Webscrapping

In [ ]:
#Coletando a data do dia
data_hoje = datetime.datetime.now(ZoneInfo("America/Sao_Paulo")).date()
data_formatada = data_hoje.strftime("%d/%m/%Y")

# Lista de nomes que você quer capturar (normalizados)
nomes_alvo = ["livelo", "shopping livelo", "esfera", "smiles", "azul", "shopping latam", "latam", "pass", "LATAM Pass", "Shopping Livelo"]

#Buscando a hora atual
agora_brasilia = datetime.datetime.now(ZoneInfo("America/Sao_Paulo"))

hora_atual = int(agora_brasilia.strftime("%H"))

def coletar_promocoes(url, parceiro, data_formatada, nomes_alvo):
    pagina = requests.get(url)
    dados_pagina = BeautifulSoup(pagina.text, 'html.parser')

    # --- Captura dos nomes ---
    list_nomes_clubes = []
    for sp in dados_pagina.select("div.d-grid span"):
        nome = sp.get_text(strip=True)
        nome_norm = nome.lower()

        if nome_norm not in nomes_alvo:
            continue

        # Emojis
        if nome_norm in ("livelo", "shopping livelo"):
            nome = f"🟣 {nome}"
        elif nome_norm == "esfera":
            nome = f"🔴 {nome}"
        elif nome_norm == "smiles":
            nome = f"🟠 {nome}"
        elif nome_norm in ("azul", "shopping latam", "latam"):
            nome = f"🔵 {nome}"
        else:
            nome = nome

        list_nomes_clubes.append(nome)

    # --- Captura da pontuação ---
    list_pontuacao_clubes = []
    tabela_pm = dados_pagina.find('th', string=lambda s: s and 'Pontos e Milhas' in s)
    if tabela_pm:
        table = tabela_pm.find_parent('table')
        for tr in table.select('tbody tr'):
            nome_el = tr.select_one('td .d-grid span')
            if not nome_el:
                continue
            nome_norm = nome_el.get_text(strip=True).lower()

            if nome_norm not in nomes_alvo:
                continue

            tds = tr.find_all('td')
            if len(tds) < 2:
                continue

            texto_ganho = tds[-1].get_text(" ", strip=True)
            m = re.search(r'(\d+(?:[.,]\d+)?)\s*pt', texto_ganho, flags=re.I)
            if not m:
                continue

            valor = m.group(1).replace(',', '.')
            val_float = float(valor)
            val_final = int(val_float) if val_float.is_integer() else val_float

            list_pontuacao_clubes.append(val_final)

    # Monta DataFrame
    df = pd.DataFrame({
        "nome_clube": list_nomes_clubes,
        "pontuacao_clube": list_pontuacao_clubes
    })

    # --- Ajuste de pontuação e análise ---
    list_analise = []
    list_pontuacao_ajustada = []
    for index, row in df.iterrows():
        nome_norm = row["nome_clube"].lower()

        if ("livelo" not in nome_norm) and ("esfera" not in nome_norm):
            pontuacao_ajustada = row["pontuacao_clube"] / 1.3
        else:
            pontuacao_ajustada = row["pontuacao_clube"]

        #Analisando a promoção
        if pontuacao_ajustada >= 10:
          d_analise = '⭐⭐⭐⭐⭐ Nível 5 (Excelente / Raro) 😏: Promoção rara, aproveite sem medo!'
        elif pontuacao_ajustada >= 8:
          d_analise = '⭐⭐⭐⭐ Nível 4 (Muito Bom) 😎: Ótima Promoção, daqui pra cima já vale muito!'
        elif pontuacao_ajustada >= 5:
          d_analise = '⭐⭐⭐ Nível 3 (Bom) 😉: Bom momento para potencializar compras planejadas, mas se puder aguardar, tem coisa melhor!'
        elif pontuacao_ajustada >= 3:
          d_analise = '⭐⭐ Nível 2 (Mediano) 🧐: Dá pra usar se você realmente já iria comprar, mas é melhor aguardar algo melhor ;)'
        else:
          d_analise = '⭐ Nível 1 (Ruim) 😡: Sugiro aguardar algo melhor'

        list_analise.append(d_analise)
        list_pontuacao_ajustada.append(pontuacao_ajustada)

    df["analise_pontuacao"] = list_analise
    df["pontuacao_ajustada"] = list_pontuacao_ajustada
    df["parceiro"] = parceiro
    df["data_coleta_promocao"] = data_formatada

    return df

### Controle de Envios (substitui as janelas de horário)

In [ ]:
# =========================================
# 🔢 CONTROLE DE ENVIOS (substitui as janelas de horário)
# =========================================
# Em vez de disparar cada bloco com base na hora atual (hora_atual),
# cada execução consulta a planilha "Controle de Envios Grupo" para saber
# quantos envios já foram feitos HOJE e assume o próximo número da
# sequência (1º envio do dia, 2º envio do dia, etc).
#
# Isso resolve o problema do GitHub Actions atrasar e o cron cair fora
# da janela de horário esperada: se um gatilho atrasar ou for perdido,
# o próximo gatilho que rodar simplesmente assume o próximo número da
# sequência, em vez de pular aquele bloco para sempre naquele dia.
#
# IMPORTANTE: a planilha "Controle de Envios Grupo" é um arquivo separado
# da planilha de Feriados/Eventos/Base de Dados (SHEET_ID). É necessário:
# 1) Criar o secret SHEET_ID_CONTROLE_ENVIOS no GitHub Actions com o ID
#    dessa planilha (o trecho da URL entre /d/ e /edit).
# 2) Compartilhar a planilha com o e-mail da service account (campo
#    "client_email" dentro do secret GOOGLE_CREDENTIALS) como Editor.
# A aba chamada "Envios_Geral_Grupo" (não precisa ser a primeira — é
# buscada pelo nome, já que a planilha também tem "Análise", "Envios" e
# "Ranking_dia") deve ter o cabeçalho: data | data_hora_execucao | numero_envio | status
#
# Cada envio pode ser tentado até MAX_TENTATIVAS_POR_ENVIO vezes no mesmo
# dia. Só avança pro próximo número quando o atual teve sucesso ("OK") ou
# esgotou as tentativas (aí desiste dele só naquele dia e segue em frente,
# pra uma falha pontual não travar a sequência inteira).

import json
import gspread
from oauth2client.service_account import ServiceAccountCredentials

# =========================================================================
# 🔁 RETRY AUTOMÁTICO PARA CHAMADAS AO GOOGLE SHEETS
# =========================================================================
# A API do Google Sheets ocasionalmente retorna erros transitórios (ex:
# 503 "service unavailable", ou 429 de limite de requisições) que não têm
# relação com o nosso código — é só instabilidade momentânea do lado do
# Google. Em vez de adicionar retry em cada uma das dezenas de chamadas
# gspread espalhadas pelo notebook (conectar, ler aba, gravar linha,
# apagar linha, limpar aba...), aplicamos o retry UMA VEZ SÓ, no método
# HTTP interno que o gspread usa por baixo dos panos — assim toda chamada
# fica protegida automaticamente, sem precisar mexer em cada uma nem
# lembrar de proteger chamadas novas que a gente adicionar no futuro.
import time as _time_module
from gspread.http_client import HTTPClient as _GspreadHTTPClient
from gspread.exceptions import APIError as _GspreadAPIError

_request_original_gspread = _GspreadHTTPClient.request

def _request_com_retry(self, *args, max_tentativas=4, espera_inicial=2, **kwargs):
    for tentativa in range(1, max_tentativas + 1):
        try:
            return _request_original_gspread(self, *args, **kwargs)
        except _GspreadAPIError as e:
            status_code = getattr(getattr(e, "response", None), "status_code", None)
            # Só vale a pena tentar de novo em erros transitórios (5xx do
            # lado do Google, ou 429 de limite de requisições) — erros
            # como 403/404 (permissão, aba não encontrada) não se
            # resolvem tentando de novo, então propaga na hora.
            transitorio = status_code is not None and (status_code >= 500 or status_code == 429)
            if not transitorio or tentativa == max_tentativas:
                raise
            espera = espera_inicial * (2 ** (tentativa - 1))
            print(f"⚠️ Erro transitório do Google Sheets ({status_code}) — tentativa {tentativa}/{max_tentativas}, aguardando {espera}s...")
            _time_module.sleep(espera)

_GspreadHTTPClient.request = _request_com_retry

# =========================================================================
# 🌙 PROTEÇÃO: NUNCA ENVIAR NADA ENTRE 22h E 06h59 (horário de Brasília)
# =========================================================================
# Intercepta toda chamada requests.post feita pro domínio do ZapperHub —
# não importa qual célula chamou, nem se foi um gatilho normal, atrasado
# ou disparado manualmente (workflow_dispatch): se o horário atual em
# Brasília estiver dentro da janela de silêncio, a mensagem NÃO é enviada
# de verdade. Devolve uma resposta com status diferente de 200, então o
# código de cada célula (que já checa "if response.status_code == 200")
# trata isso como uma falha normal — ou seja, o envio fica marcado como
# "Erro" e é tentado de novo automaticamente na próxima execução dentro
# do horário permitido, em vez de ser silenciosamente dado como enviado.
HORA_INICIO_SILENCIO = 22  # a partir das 22h
HORA_FIM_SILENCIO = 7      # até 06h59 (permite de novo a partir das 07h)

_post_original_requests = requests.post

def _post_com_protecao_horario(url_chamada, *args, **kwargs):
    eh_zapperapi = "api.zapperapi.com" in url_chamada
    hora_agora = agora_brasilia.hour
    dentro_do_silencio = hora_agora >= HORA_INICIO_SILENCIO or hora_agora < HORA_FIM_SILENCIO

    if eh_zapperapi and dentro_do_silencio:
        print(f"🌙 Envio bloqueado — {hora_agora}h está dentro do horário de silêncio (22h-06h59, Brasília). Mensagem NÃO enviada, será tentada de novo depois.")
        resposta_bloqueada = requests.Response()
        resposta_bloqueada.status_code = 425  # "Too Early" — nunca é 200, então o código chamador trata como falha
        resposta_bloqueada._content = json.dumps({
            "erro": "Envio bloqueado pelo horário de silêncio (22h-06h59, horário de Brasília)"
        }).encode("utf-8")
        return resposta_bloqueada

    return _post_original_requests(url_chamada, *args, **kwargs)

requests.post = _post_com_protecao_horario

# 🐙 VERSÃO GITHUB ACTIONS (ativa)
SHEET_ID = os.environ["SHEET_ID"]
SHEET_ID_CONTROLE_ENVIOS = os.environ["SHEET_ID_CONTROLE_ENVIOS"]

MAX_TENTATIVAS_POR_ENVIO = 3

# 🖥️ VERSÃO LOCAL (Colab)
#SHEET_ID = ler_config_local("sheet_id.txt")
#SHEET_ID_CONTROLE_ENVIOS = ler_config_local("sheet_id_controle_envios.txt")

# Ordem de envio do dia -> nome do bloco (usado só nos logs, não é gravado na planilha)
NOME_BLOCO_POR_NUMERO = {
    1: "Perfumaria e Cosméticos",
    2: "Roupas e Calçados Esportivos",
    3: "Moda",
    4: "Beleza e Cosméticos",
    5: "Varejo",
    6: "Transferências Bonificadas",
    7: "Jabares Noturnos",
}

def conectar_google_sheets():
    """Conecta na planilha de Feriados/Eventos/Base de Dados (SHEET_ID)"""
    scope = [
        "https://spreadsheets.google.com/feeds",
        "https://www.googleapis.com/auth/drive"
    ]

    # 🐙 VERSÃO GITHUB ACTIONS (ativa)
    creds_dict = json.loads(os.environ["GOOGLE_CREDENTIALS"])
    creds = ServiceAccountCredentials.from_json_keyfile_dict(creds_dict, scope)

    # 🖥️ VERSÃO LOCAL (Colab) — descomenta e comenta o bloco acima para testar local
    #from google.colab import files
    #uploaded = files.upload()  # faz upload do credenciais.json
    #creds = ServiceAccountCredentials.from_json_keyfile_name("credenciais.json", scope)

    client_gs = gspread.authorize(creds)
    spreadsheet = client_gs.open_by_key(SHEET_ID)
    return spreadsheet

def conectar_spreadsheet_controle_envios():
    """Conecta na planilha 'Controle de Envios Grupo' (SHEET_ID_CONTROLE_ENVIOS)
    e retorna o Spreadsheet inteiro — usado pra acessar qualquer aba dela
    (Envios_Geral_Grupo, Análise, Envios, Ranking_dia etc) pelo nome."""
    scope = [
        "https://spreadsheets.google.com/feeds",
        "https://www.googleapis.com/auth/drive"
    ]

    creds_dict = json.loads(os.environ["GOOGLE_CREDENTIALS"])
    creds = ServiceAccountCredentials.from_json_keyfile_dict(creds_dict, scope)
    client_gs = gspread.authorize(creds)

    return client_gs.open_by_key(SHEET_ID_CONTROLE_ENVIOS)

def carregar_feriados(spreadsheet):
    """Carrega a aba Feriados e retorna um dict {date: nome_feriado}"""
    aba = spreadsheet.worksheet("Feriados")
    dados = aba.get_all_values()
    feriados = {}
    for row in dados[1:]:  # pula o cabeçalho
        if len(row) >= 2 and row[0] and row[1]:
            try:
                data = datetime.datetime.strptime(row[0].strip(), "%d/%m/%Y").date()
                feriados[data] = row[1].strip()
            except ValueError:
                continue
    return feriados

def carregar_eventos(spreadsheet):
    """Carrega a aba Eventos e retorna um dict {date: nome_evento}"""
    aba = spreadsheet.worksheet("Eventos")
    dados = aba.get_all_values()
    eventos = {}
    for row in dados[1:]:  # pula o cabeçalho
        if len(row) >= 2 and row[0] and row[1]:
            try:
                data = datetime.datetime.strptime(row[0].strip(), "%d/%m/%Y").date()
                eventos[data] = row[1].strip()
            except ValueError:
                continue
    return eventos

def gravar_no_sheets(spreadsheet, dados_row):
    """Adiciona uma nova linha na aba Base de Dados"""
    aba = spreadsheet.worksheet("Base de Dados")
    aba.append_row(dados_row, value_input_option="USER_ENTERED")
    print("✅ Linha gravada no Google Sheets!")

# =========================================================================
# 🏆 RANKING DO DIA (aba "Ranking_dia")
# =========================================================================
# Diferente do "Base de Dados", essa aba NÃO guarda histórico entre dias —
# ela é zerada no primeiro envio do dia (envio 1) e vai sendo preenchida
# pelas categorias de lojas (envios 1 a 5) com a pontuação de cada
# promoção. No envio 7, lê tudo que está lá (só as promoções de hoje,
# já que a aba começou vazia hoje) e manda o top 5 como ranking pro grupo.
COLUNAS_RANKING_DIA = ["data", "categoria", "parceiro", "nome_clube", "pontuacao_clube", "pontuacao_ajustada", "analise_pontuacao"]

def resetar_ranking_dia(aba):
    """Limpa a aba Ranking_dia inteira e recria o cabeçalho — chamado só
    no envio 1 (o primeiro do dia), antes de qualquer promoção ser
    enviada, pra começar o dia do zero."""
    aba.clear()
    aba.append_row(COLUNAS_RANKING_DIA, value_input_option="USER_ENTERED")
    print("🧹 Aba 'Ranking_dia' zerada para o novo dia.")

def registrar_promocoes_ranking(aba, df, categoria, data_formatada):
    """Grava cada linha de um df_final (com a pontuação já calculada) na
    aba Ranking_dia (usada pro ranking do envio 7) e também na aba
    Ranking_historico — essa nunca é limpa, guarda o histórico completo de
    todas as promoções registradas, dia após dia (Ranking_dia começa
    vazia de novo a cada dia, no envio 1)."""
    for _, linha in df.iterrows():
        linha_dados = [
            data_formatada,
            categoria,
            linha.get("parceiro", ""),
            linha.get("nome_clube", ""),
            linha.get("pontuacao_clube", ""),
            linha.get("pontuacao_ajustada", ""),
            linha.get("analise_pontuacao", ""),
        ]
        aba.append_row(linha_dados, value_input_option="USER_ENTERED")
        aba_ranking_historico.append_row(linha_dados, value_input_option="USER_ENTERED")


def garantir_cabecalho_ranking_historico(aba):
    """Escreve o cabeçalho na aba Ranking_historico só se ela ainda
    estiver vazia (primeiro uso) — diferente de resetar_ranking_dia, NUNCA
    limpa o conteúdo já existente."""
    if not aba.row_values(1):
        aba.append_row(COLUNAS_RANKING_DIA, value_input_option="USER_ENTERED")
        print("🗂️ Cabeçalho criado na aba 'Ranking_historico'.")

def carregar_status_envios_hoje(planilha_controle, data_formatada):
    """Lê a planilha de controle e retorna, para o dia de hoje, quantas
    tentativas cada número de envio já teve e se alguma foi bem-sucedida.
    Retorna {numero_envio: {"tentativas": N, "sucesso": True/False}}"""
    dados = planilha_controle.get_all_values()
    status = {}
    for row in dados[1:]:  # pula o cabeçalho
        if len(row) >= 3 and row[0].strip() == data_formatada:
            try:
                numero = int(row[2])
            except (ValueError, IndexError):
                continue
            resultado_status = row[3].strip() if len(row) > 3 else ""
            if numero not in status:
                status[numero] = {"tentativas": 0, "sucesso": False}
            status[numero]["tentativas"] += 1
            if resultado_status == "OK":
                status[numero]["sucesso"] = True
    return status

def determinar_numero_envio_hoje(status_envios_hoje, total_envios):
    """Percorre os envios em ordem e retorna o primeiro que ainda não foi
    resolvido (nem teve sucesso, nem esgotou as tentativas)."""
    for n in range(1, total_envios + 1):
        info = status_envios_hoje.get(n, {"tentativas": 0, "sucesso": False})
        if info["sucesso"]:
            continue
        if info["tentativas"] >= MAX_TENTATIVAS_POR_ENVIO:
            continue
        return n
    return total_envios + 1

def registrar_tentativa_envio(planilha_controle, numero_envio, status):
    """Registra o resultado de uma tentativa de envio ('OK' ou 'Erro')."""
    agora_str = agora_brasilia.strftime("%d/%m/%Y %H:%M:%S")
    linha = [data_formatada, agora_str, numero_envio, status]
    planilha_controle.append_row(linha, value_input_option="USER_ENTERED")
    print(f"📋 Envio {numero_envio} registrado como '{status}' às {agora_str}.")

# ── Conecta e descobre qual é o envio da vez ──
print("📊 Conectando ao Google Sheets...")
spreadsheet = conectar_google_sheets()
spreadsheet_controle_envios = conectar_spreadsheet_controle_envios()
planilha_controle = spreadsheet_controle_envios.worksheet("Envios_Geral_Grupo")
aba_ranking_dia = spreadsheet_controle_envios.worksheet("Ranking_dia")
aba_ranking_historico = spreadsheet_controle_envios.worksheet("Ranking_historico")
garantir_cabecalho_ranking_historico(aba_ranking_historico)

status_envios_hoje = carregar_status_envios_hoje(planilha_controle, data_formatada)
numero_envio_hoje = determinar_numero_envio_hoje(status_envios_hoje, len(NOME_BLOCO_POR_NUMERO))

# Flag global: cada bloco de envio ajusta pra False se algum POST falhar.
# No final do grupo (última célula daquele número), o resultado é
# registrado na planilha via registrar_tentativa_envio.
envio_atual_ok = True

if numero_envio_hoje <= len(NOME_BLOCO_POR_NUMERO):
    nome_bloco_hoje = NOME_BLOCO_POR_NUMERO[numero_envio_hoje]
    tentativa_atual = status_envios_hoje.get(numero_envio_hoje, {"tentativas": 0})["tentativas"] + 1
    print(f"➡️ Envio nº {numero_envio_hoje} de hoje: {nome_bloco_hoje} (tentativa {tentativa_atual}/{MAX_TENTATIVAS_POR_ENVIO})")
else:
    nome_bloco_hoje = None
    print(f"✅ Todos os envios de hoje já foram resolvidos (enviados ou tentativas esgotadas).")

📊 Conectando ao Google Sheets...
➡️ Esse é o envio nº 6 de hoje: Transferências Bonificadas
📋 Envio 6 registrado na planilha às 12/07/2026 11:39:09.


### Mensagem Bom dia

In [ ]:
#Criando o texto de bom dia

dia_semana = agora_brasilia.strftime("%A")  # Nome do dia em inglês
hora_atual_saudacao = agora_brasilia.hour   # horário de Brasília

if dia_semana == "Monday":
    txt_dia_semana = "segundou! 😴"
elif dia_semana == "Tuesday":
    txt_dia_semana = "terçou!"
elif dia_semana == "Wednesday":
    txt_dia_semana = "quartou!"
elif dia_semana == "Thursday":
    txt_dia_semana = "quintou! 🚀"
elif dia_semana == "Friday":
    txt_dia_semana = "sextou! 🔥"
elif dia_semana == "Saturday":
    txt_dia_semana = "sabadou! 😎"
elif dia_semana == "Sunday":
    txt_dia_semana = "domingou! ☀️"
else:
    txt_dia_semana = "vocês estão bem?"

# Saudação dinâmica de acordo com o horário de Brasília. Antes a mensagem
# assumia que era sempre de manhã (fazia sentido quando esse envio era
# fixo na janela das 7h-8h). Agora que a ordem de envio é quem manda
# (numero_envio_hoje), esse bloco pode acabar rodando à tarde ou à noite
# se algum gatilho atrasar — então a saudação acompanha o horário real.
if 5 <= hora_atual_saudacao < 12:
    saudacao = "Bom dia"
    emoji_saudacao = "☀️"
    encerramento_saudacao = "Ótimo dia a todos!"
elif 12 <= hora_atual_saudacao < 18:
    saudacao = "Boa tarde"
    emoji_saudacao = "🌤️"
    encerramento_saudacao = "Ótima tarde a todos!"
else:
    saudacao = "Boa noite"
    emoji_saudacao = "🌙"
    encerramento_saudacao = "Ótima noite a todos!"

texto_bom_dia = f"""
{emoji_saudacao} *{saudacao} milheiros, {txt_dia_semana}*

Antes de começar, segue o nosso critério de análise para você aproveitar as melhores promoções no acúmulo de milhas:

Nós avaliamos cada promoção em 5 níveis, com base no retorno de milhas/pontos:

• *Nível 1 (Ruim) ⭐* Menos que 3 pontos → Sugiro aguardar algo melhor.
• *Nível 2 (Mediano) ⭐⭐* 3 pontos ou mais → Use só se você já iria comprar de qualquer forma.
• *Nível 3 (Bom) ⭐⭐⭐* 5 pontos ou mais → Bom para compras planejadas, mas ainda pode melhorar.
• *Nível 4 (Muito Bom) ⭐⭐⭐⭐* 8 pontos ou mais → Ótima promoção, daqui pra cima já vale muito a pena!
• *Nível 5 (Excelente / Raro) ⭐⭐⭐⭐⭐* 10 pontos ou mais → Promoção rara, dessas que não aparecem toda hora. Aproveite sem medo!

Quanto maior o nível, melhor o custo-benefício para acumular milhas/pontos.

Obs: Em promoções diretamente nas cias aéreas, nos aplicamos uma divisão de 1,30 para igualar as milhas ao pontos (considerando
uma transferência bonificada de 30%)!

{encerramento_saudacao} Bora pra cima! 🚀

Quer mais dicas de milhas, cartões e viagens? Dá uma olhada no site: https://murilloborges.com.br/wa
"""

## Grupo de Perfumaria (Envio 1)

### Loja Natura

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 1:
    df_natura = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-natura/",
        parceiro="Natura",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_natura)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Boticário

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 1:
    df_boticario = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-boticario/",
        parceiro="Boticário",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_boticario)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Eudora

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 1:
    df_eudora = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-eudora/",
        parceiro="Eudora",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_eudora)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Enviando os dados via Whatsapp

In [ ]:
if numero_envio_hoje == 1:
  # É o primeiro envio do dia — zera a aba Ranking_dia antes de mandar
  # qualquer promoção, pra começar a acumular o dia do zero.
  resetar_ranking_dia(aba_ranking_dia)

  # Concatenando os dataframes
  df_final = pd.concat([df_natura, df_boticario, df_eudora], ignore_index=True)
  df_final

  # Registra as promoções desta categoria na aba Ranking_dia (usado pro
  # ranking do top 5 que sai no envio 7).
  registrar_promocoes_ranking(aba_ranking_dia, df_final, "Perfumaria e Cosméticos", data_formatada)

  # Criando a mensagem do grupo de Beleza e Cosméticos
  msg = ""
  msg_titulo = f"💄 *Resumo das Promoções da Categoria de Perfumaria e Cosméticos em {data_formatada}: 👇*"

  for index, row in df_final.iterrows():
    msg += textwrap.dedent(f"""
  *Parceiro:* {row["parceiro"]}
  *Programa de Fidelidade:* {row["nome_clube"]}
  *Pontuação:* {row["pontuacao_clube"]}
  *Análise da Promoção:* {row["analise_pontuacao"]}

  {"-"*30}""").strip()

  msg_grupo = msg_titulo + "\n""\n" + msg

  # Mensagem de Bom dia
  message_bom_dia = texto_bom_dia
  message_promo = msg_grupo

  # Payload
  payload = {
      "jid": id_grupo_envio,
      "message": message_bom_dia,
  }

  #Enviando a mensagem de bom dia
  response = requests.post(url, json=payload, headers=headers)

  # Resposta
  if response.status_code == 200:
      print("Mensagem de bom dia enviada com sucesso para o grupo!")
  else:
      print(f"Erro ao enviar mensagem: {response.status_code} - {response.text}")
      envio_atual_ok = False


  # Aguardar 2 minutos
  time.sleep(120)

  # Payload
  payload = {
      "jid": id_grupo_envio, #group_jid,
      "message": message_promo,
  }

  #Enviando as promoções
  response = requests.post(url, json=payload, headers=headers)

  # Resposta
  if response.status_code == 200:
      print("Mensagem com as promoções enviada com sucesso para o grupo!")
  else:
      print(f"Erro ao enviar mensagem: {response.status_code} - {response.text}")
      envio_atual_ok = False

  registrar_tentativa_envio(planilha_controle, numero_envio_hoje, "OK" if envio_atual_ok else "Erro")

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


## Grupo de Suplementos / Fitness (Envio 1)

In [ ]:
if numero_envio_hoje == 1:

  # Aguardar 2 minutos
  time.sleep(120)

  PARCEIROS_SUPLEMENTOS_LIVELO = ['Max Titanium', 'Probiótica']
  PARCEIROS_SUPLEMENTOS_ESFERA = ['Max Titanium', 'Probiótica', 'Soldiers Nutrition', 'iHerb']

  itens_suplementos = []

  try:
      for p in extrair_parceiros_livelo():
          if p["parceiro"] in PARCEIROS_SUPLEMENTOS_LIVELO:
              itens_suplementos.append({
                  "parceiro": p["parceiro"],
                  "nome_clube": "🟣 Livelo",
                  "pontuacao": p["pontuacao"],
                  "link_parceiro": p["link_parceiro"],
              })
  except Exception as e:
      print(f"⚠️ Erro ao extrair parceiros de Suplementos / Fitness da Livelo: {e}")

  try:
      for p in extrair_parceiros_esfera():
          if p["parceiro"] in PARCEIROS_SUPLEMENTOS_ESFERA:
              itens_suplementos.append({
                  "parceiro": p["parceiro"],
                  "nome_clube": "🔴 Esfera",
                  "pontuacao": p["pontuacao"],
                  "link_parceiro": p["link_parceiro"],
              })
  except Exception as e:
      print(f"⚠️ Erro ao extrair parceiros de Suplementos / Fitness da Esfera: {e}")

  if itens_suplementos:
      # Registra na aba Ranking_dia (+ Ranking_historico), igual as outras
      # categorias, pra também concorrer no ranking do top 5 do envio 7.
      df_suplementos = pd.DataFrame([
          {
              "parceiro": item["parceiro"],
              "nome_clube": item["nome_clube"],
              "pontuacao_clube": item["pontuacao"],
              "pontuacao_ajustada": item["pontuacao"],
              "analise_pontuacao": analisar_pontuacao_extracao(item["pontuacao"]),
          }
          for item in itens_suplementos
      ])
      registrar_promocoes_ranking(aba_ranking_dia, df_suplementos, "Suplementos / Fitness", data_formatada)

      # Criando a mensagem do grupo de Suplementos / Fitness
      msg = ""
      msg_titulo = f"💪 *Promoções da Categoria de Suplementos / Fitness em {data_formatada}: 👇*"

      for item in itens_suplementos:
          msg += textwrap.dedent(f"""
      *Parceiro:* {item["parceiro"]}
      *Programa de Fidelidade:* {item["nome_clube"]}
      *Pontuação:* {formatar_pontuacao_extracao(item["pontuacao"])}
      *Link parceiro:* {item["link_parceiro"]}
      *Análise da Promoção:* {analisar_pontuacao_extracao(item["pontuacao"])}

      {"-"*30}""").strip()

      msg_grupo = msg_titulo + "\n\n" + msg

      payload = {
          "jid": id_grupo_envio,
          "message": msg_grupo,
      }

      response = requests.post(url, json=payload, headers=headers)

      if response.status_code == 200:
          print("Mensagem de Suplementos / Fitness enviada com sucesso para o grupo!")
      else:
          print(f"Erro ao enviar mensagem: {response.status_code} - {response.text}")
          envio_atual_ok = False
  else:
      print("⚠️ Nenhum parceiro de suplementos / fitness encontrado nesta execução (Livelo/Esfera) - nada enviado.")

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")


##

## Grupo de Roupas e Calçados Esportivos (Envio 2) GRUPO OK

### Loja Centauro

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 2:
    df_centauro = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-centauro/",
        parceiro="Centauro",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_centauro)

    # 🔔 Alerta pessoal: Centauro pagando 6 pontos ou mais em qualquer clube
    for _, linha_centauro in df_centauro[df_centauro["pontuacao_clube"] >= 6].iterrows():
        enviar_alerta_pessoal(
            "Centauro bonificada",
            f"Centauro está pagando {linha_centauro['pontuacao_clube']} pontos por real em {linha_centauro['nome_clube']}!"
        )
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja NetShoes

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 2:
    df_netshoes = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-netshoes/",
        parceiro="Netshoes",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_netshoes)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja New Balance

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 2:
    df_new_balance = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-new-balance/",
        parceiro="New Balance",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_new_balance)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Nike

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 2:
    df_nike = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-nike/",
        parceiro="Nike",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_nike)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Adidas

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 2:
    df_adidas = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-adidas/",
        parceiro="Adidas",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_adidas)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Enviando os dados via Whatsapp

### Jabar Guias de Viagem (Envio 2, Quarta-feira e Sábado)

In [ ]:
if numero_envio_hoje == 2 and dia_semana in ("Wednesday", "Saturday"):

  # Aguardar 1 minuto
  time.sleep(60)

  drive_id = "1kvhTEB0PZPf_Ib-2lM2Kl2YiprMiqYPh"
  img_url = f"https://drive.google.com/uc?export=view&id={drive_id}"

  message_promo = textwrap.dedent("""
  📖 *Planejando uma viagem e não sabe por onde começar?*

  Preparei guias completos com dicas de roteiro, onde ficar, o que fazer e como economizar em cada destino. É tudo que eu mesmo uso quando estou organizando as minhas próprias viagens.

  É o atalho pra você economizar tempo de pesquisa e já sair com um plano na mão ✈️

  👉 Acesse todos os guias aqui:
  https://murilloborges.com.br/guias/
  """).strip()

  payload = {
      "jid": id_grupo_envio,
      "mediaType": "image",
      "mimetype": "image/jpeg",
      "media": img_url,
      "caption": message_promo,
      "filename": "analise.jpg"
  }

  response = requests.post(url_media, json=payload, headers=headers)

  print(response.json())
  if response.status_code != 200:
      envio_atual_ok = False

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")


### Jabar Produtos que eu indico para viagens (Envio 2, Domingo)

In [ ]:
if numero_envio_hoje == 2 and dia_semana == "Sunday":

  # Aguardar 1 minuto
  time.sleep(60)

  drive_id = "13zwyjYxvXiHS7or9THw2-UZnSGLpeUDX"
  img_url = f"https://drive.google.com/uc?export=view&id={drive_id}"

  message_promo = textwrap.dedent("""
  🎒 *Quer saber quais produtos eu realmente uso e levo em toda viagem?*

  Separei numa página só os itens que fazem diferença na hora de viajar: bagagem, eletrônicos, acessórios e tudo que já testei na prática e recomendo de verdade.

  Nada de indicação genérica. É uma lista pensada pra facilitar sua vida antes de arrumar a mala 🧳

  👉 Veja a lista completa aqui:
  https://murilloborges.com.br/links-produtos/
  """).strip()

  payload = {
      "jid": id_grupo_envio,
      "mediaType": "image",
      "mimetype": "image/jpeg",
      "media": img_url,
      "caption": message_promo,
      "filename": "analise.jpg"
  }

  response = requests.post(url_media, json=payload, headers=headers)

  print(response.json())
  if response.status_code != 200:
      envio_atual_ok = False

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")


In [ ]:
if numero_envio_hoje == 2:
  # Concatenando os dataframes
  df_final = pd.concat([df_centauro, df_netshoes, df_new_balance, df_nike, df_adidas], ignore_index=True)
  df_final

  # Registra as promoções desta categoria na aba Ranking_dia (usado pro
  # ranking do top 5 que sai no envio 7).
  registrar_promocoes_ranking(aba_ranking_dia, df_final, "Roupas e Calçados Esportivos", data_formatada)

  # Criando a mensagem do grupo de Beleza e Cosméticos
  msg = ""
  msg_titulo = f"👟🏓 *Promoções da Categoria de Roupas e Calçados Esportivos em {data_formatada}: 👇*"

  for index, row in df_final.iterrows():
    msg += textwrap.dedent(f"""
  *Parceiro:* {row["parceiro"]}
  *Programa de Fidelidade:* {row["nome_clube"]}
  *Pontuação:* {row["pontuacao_clube"]}
  *Análise da Promoção:* {row["analise_pontuacao"]}

  {"-"*30}""").strip()

  msg_grupo = msg_titulo + "\n\n" + msg

  # Mensagem
  message_promo = msg_grupo

  # Payload
  payload = {
      "jid": id_grupo_envio, #group_jid,
      "message": message_promo,
  }

  #Enviando as promoções
  response = requests.post(url, json=payload, headers=headers)

  # Resposta
  if response.status_code == 200:
      print("Mensagem com as promoções enviada com sucesso para o grupo!")
  else:
      print(f"Erro ao enviar mensagem: {response.status_code} - {response.text}")
      envio_atual_ok = False

  registrar_tentativa_envio(planilha_controle, numero_envio_hoje, "OK" if envio_atual_ok else "Erro")

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")


Não é o envio da vez hoje (ou este envio já foi feito).


## Grupo de Café (Envio 2)

In [ ]:
if numero_envio_hoje == 2:

  # Aguardar 2 minutos
  time.sleep(120)

  PARCEIROS_CAFE_LIVELO = ['Coffee Mais', 'Assinatura Coffee Mais', 'Café Orfeu']
  PARCEIROS_CAFE_ESFERA = ['Coffee++', "Café L'or", 'Café Store', 'Dolce Gusto', 'Nespresso', 'Pilão']

  itens_cafe = []

  try:
      for p in extrair_parceiros_livelo():
          if p["parceiro"] in PARCEIROS_CAFE_LIVELO:
              itens_cafe.append({
                  "parceiro": p["parceiro"],
                  "nome_clube": "🟣 Livelo",
                  "pontuacao": p["pontuacao"],
                  "link_parceiro": p["link_parceiro"],
              })
  except Exception as e:
      print(f"⚠️ Erro ao extrair parceiros de Café da Livelo: {e}")

  try:
      for p in extrair_parceiros_esfera():
          if p["parceiro"] in PARCEIROS_CAFE_ESFERA:
              itens_cafe.append({
                  "parceiro": p["parceiro"],
                  "nome_clube": "🔴 Esfera",
                  "pontuacao": p["pontuacao"],
                  "link_parceiro": p["link_parceiro"],
              })
  except Exception as e:
      print(f"⚠️ Erro ao extrair parceiros de Café da Esfera: {e}")

  if itens_cafe:
      # Registra na aba Ranking_dia (+ Ranking_historico), igual as outras
      # categorias, pra também concorrer no ranking do top 5 do envio 7.
      df_cafe = pd.DataFrame([
          {
              "parceiro": item["parceiro"],
              "nome_clube": item["nome_clube"],
              "pontuacao_clube": item["pontuacao"],
              "pontuacao_ajustada": item["pontuacao"],
              "analise_pontuacao": analisar_pontuacao_extracao(item["pontuacao"]),
          }
          for item in itens_cafe
      ])
      registrar_promocoes_ranking(aba_ranking_dia, df_cafe, "Café", data_formatada)

      # Criando a mensagem do grupo de Café
      msg = ""
      msg_titulo = f"☕ *Promoções da Categoria de Café em {data_formatada}: 👇*"

      for item in itens_cafe:
          msg += textwrap.dedent(f"""
      *Parceiro:* {item["parceiro"]}
      *Programa de Fidelidade:* {item["nome_clube"]}
      *Pontuação:* {formatar_pontuacao_extracao(item["pontuacao"])}
      *Link parceiro:* {item["link_parceiro"]}
      *Análise da Promoção:* {analisar_pontuacao_extracao(item["pontuacao"])}

      {"-"*30}""").strip()

      msg_grupo = msg_titulo + "\n\n" + msg

      payload = {
          "jid": id_grupo_envio,
          "message": msg_grupo,
      }

      response = requests.post(url, json=payload, headers=headers)

      if response.status_code == 200:
          print("Mensagem de Café enviada com sucesso para o grupo!")
      else:
          print(f"Erro ao enviar mensagem: {response.status_code} - {response.text}")
          envio_atual_ok = False
  else:
      print("⚠️ Nenhum parceiro de café encontrado nesta execução (Livelo/Esfera) - nada enviado.")

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")


## Grupo de Moda (Envio 3)

### Loja Renner

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 3:
    df_renner = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-lojas-renner/",
        parceiro="Lojas Renner",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_renner)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Riachuelo

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 3:
    df_riachuelo = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-riachuelo/",
        parceiro="Riachuelo",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_riachuelo)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja CEA

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 3:
    df_cea = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-cea/",
        parceiro="CEA",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_cea)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Hering

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 3:
    df_hering = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-hering/",
        parceiro="CEA",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_hering)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Dafiti

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 3:
    df_dafiti = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-dafiti/",
        parceiro="Dafiti",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_dafiti)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Enviando os dados via Whatsapp

In [ ]:
if numero_envio_hoje == 3:

  # Concatenando os dataframes
  df_final = pd.concat([df_renner, df_riachuelo, df_cea, df_hering, df_dafiti], ignore_index=True)

  # Registra as promoções desta categoria na aba Ranking_dia (usado pro
  # ranking do top 5 que sai no envio 7).
  registrar_promocoes_ranking(aba_ranking_dia, df_final, "Moda", data_formatada)

  # Criando a mensagem do grupo de Beleza e Cosméticos
  msg = ""
  msg_titulo = f"👗👔 *Promoções da Categoria de Moda em {data_formatada}: 👇*"

  for index, row in df_final.iterrows():
    msg += textwrap.dedent(f"""
  *Parceiro:* {row["parceiro"]}
  *Programa de Fidelidade:* {row["nome_clube"]}
  *Pontuação:* {row["pontuacao_clube"]}
  *Análise da Promoção:* {row["analise_pontuacao"]}

  {"-"*30}""").strip()

  msg_grupo = msg_titulo + "\n\n" + msg

  # Mensagem
  message_promo = msg_grupo

  # Payload
  payload = {
      "jid": id_grupo_envio,
      "message": message_promo,
  }

  #Enviando as promoções
  response = requests.post(url, json=payload, headers=headers)

  # Resposta
  if response.status_code == 200:
      print("Mensagem com as promoções enviada com sucesso para o grupo!")
  else:
      print(f"Erro ao enviar mensagem: {response.status_code} - {response.text}")
      envio_atual_ok = False

  display(df_final)

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")


Não é o envio da vez hoje (ou este envio já foi feito).


## Grupo de Vinho (Envio 3)

In [ ]:
if numero_envio_hoje == 3:

  # Aguardar 2 minutos
  time.sleep(120)

  PARCEIROS_VINHO_LIVELO = ['Mistral', 'Divvino']
  PARCEIROS_VINHO_ESFERA = ['Mistral', 'Concha Y Toro', 'Evino', 'Freixenet', 'Grand Cru', 'Shop Vinho', 'World Wine']

  itens_vinho = []

  try:
      for p in extrair_parceiros_livelo():
          if p["parceiro"] in PARCEIROS_VINHO_LIVELO:
              itens_vinho.append({
                  "parceiro": p["parceiro"],
                  "nome_clube": "🟣 Livelo",
                  "pontuacao": p["pontuacao"],
                  "link_parceiro": p["link_parceiro"],
              })
  except Exception as e:
      print(f"⚠️ Erro ao extrair parceiros de Vinho da Livelo: {e}")

  try:
      for p in extrair_parceiros_esfera():
          if p["parceiro"] in PARCEIROS_VINHO_ESFERA:
              itens_vinho.append({
                  "parceiro": p["parceiro"],
                  "nome_clube": "🔴 Esfera",
                  "pontuacao": p["pontuacao"],
                  "link_parceiro": p["link_parceiro"],
              })
  except Exception as e:
      print(f"⚠️ Erro ao extrair parceiros de Vinho da Esfera: {e}")

  if itens_vinho:
      # Registra na aba Ranking_dia (+ Ranking_historico), igual as outras
      # categorias, pra também concorrer no ranking do top 5 do envio 7.
      df_vinho = pd.DataFrame([
          {
              "parceiro": item["parceiro"],
              "nome_clube": item["nome_clube"],
              "pontuacao_clube": item["pontuacao"],
              "pontuacao_ajustada": item["pontuacao"],
              "analise_pontuacao": analisar_pontuacao_extracao(item["pontuacao"]),
          }
          for item in itens_vinho
      ])
      registrar_promocoes_ranking(aba_ranking_dia, df_vinho, "Vinho", data_formatada)

      # Criando a mensagem do grupo de Vinho
      msg = ""
      msg_titulo = f"🍷 *Promoções da Categoria de Vinho em {data_formatada}: 👇*"

      for item in itens_vinho:
          msg += textwrap.dedent(f"""
      *Parceiro:* {item["parceiro"]}
      *Programa de Fidelidade:* {item["nome_clube"]}
      *Pontuação:* {formatar_pontuacao_extracao(item["pontuacao"])}
      *Link parceiro:* {item["link_parceiro"]}
      *Análise da Promoção:* {analisar_pontuacao_extracao(item["pontuacao"])}

      {"-"*30}""").strip()

      msg_grupo = msg_titulo + "\n\n" + msg

      payload = {
          "jid": id_grupo_envio,
          "message": msg_grupo,
      }

      response = requests.post(url, json=payload, headers=headers)

      if response.status_code == 200:
          print("Mensagem de Vinho enviada com sucesso para o grupo!")
      else:
          print(f"Erro ao enviar mensagem: {response.status_code} - {response.text}")
          envio_atual_ok = False
  else:
      print("⚠️ Nenhum parceiro de vinho encontrado nesta execução (Livelo/Esfera) - nada enviado.")

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")


## Grupo de Beleza e Cosméticos (Envio 4)

### Loja Beleza na Web

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 4:
    df_beleza_web = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-beleza-na-web/",
        parceiro="Beleza na Web",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_beleza_web)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Época Cosméticos

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 4:
    df_epoca_cosmeticos = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-epoca-cosmeticos/",
        parceiro="Época Cosméticos",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_epoca_cosmeticos)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Sephora

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 4:
    df_sephora = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-sephora/",
        parceiro="Sephora",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_sephora)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Oceane

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 4:
    df_oceane = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-oceane/",
        parceiro="Oceane",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_oceane)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Enviando os dados via Whatsapp

### Jabar Freetours Civitatis (Envio 4, Sexta-feira)

In [ ]:
if numero_envio_hoje == 4 and dia_semana == "Friday":

  # Aguardar 1 minuto
  time.sleep(60)

  drive_id = "10Yhnt2996tj0ycyQ1vHZ6xnxiog-8SrL"
  img_url = f"https://drive.google.com/uc?export=view&id={drive_id}"

  message_promo = textwrap.dedent("""
  🚶 *Já ouviu falar em free tour? É uma das formas mais baratas (e mais divertidas) de conhecer uma cidade nova!*

  São passeios guiados a pé, com guias locais, por um valor bem acessível (bem mais barato que a maioria dos passeios turísticos), mostrando os principais pontos e contando a história do lugar de um jeito leve.

  Separei mais de 100 free tours indicados, no Brasil e no mundo, prontos pra você reservar antes da sua próxima viagem 🌍

  👉 Confira todos aqui:
  https://murilloborges.com.br/links-freetours-civitatis/
  """).strip()

  payload = {
      "jid": id_grupo_envio,
      "mediaType": "image",
      "mimetype": "image/jpeg",
      "media": img_url,
      "caption": message_promo,
      "filename": "analise.jpg"
  }

  response = requests.post(url_media, json=payload, headers=headers)

  print(response.json())
  if response.status_code != 200:
      envio_atual_ok = False

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")


In [ ]:
if numero_envio_hoje == 4:
  # Concatenando os dataframes
  df_final = pd.concat([df_beleza_web, df_epoca_cosmeticos, df_sephora, df_oceane], ignore_index=True)

  # Registra as promoções desta categoria na aba Ranking_dia (usado pro
  # ranking do top 5 que sai no envio 7).
  registrar_promocoes_ranking(aba_ranking_dia, df_final, "Beleza e Cosméticos", data_formatada)

  # Criando a mensagem do grupo de Beleza e Cosméticos
  msg = ""
  msg_titulo = f"💄 *Resumo das Promoções da Categoria de Beleza e Cosméticos em {data_formatada}: 👇*"

  for index, row in df_final.iterrows():
    msg += textwrap.dedent(f"""
  *Parceiro:* {row["parceiro"]}
  *Programa de Fidelidade:* {row["nome_clube"]}
  *Pontuação:* {row["pontuacao_clube"]}
  *Análise da Promoção:* {row["analise_pontuacao"]}

  {"-"*30}""").strip()

  msg_grupo = msg_titulo + "\n""\n" + msg

  # Mensagem
  message_promo = msg_grupo

  # Payload
  payload = {
      "jid": id_grupo_envio,
      "message": message_promo,
  }

  #Enviando as promoções
  response = requests.post(url, json=payload, headers=headers)

  # Resposta
  if response.status_code == 200:
      print("Mensagem com as promoções enviada com sucesso para o grupo!")
  else:
      print(f"Erro ao enviar mensagem: {response.status_code} - {response.text}")
      envio_atual_ok = False

  display(df_final)

  registrar_tentativa_envio(planilha_controle, numero_envio_hoje, "OK" if envio_atual_ok else "Erro")

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")


Não é o envio da vez hoje (ou este envio já foi feito).


## Grupo de Mercado (Envio 4)

In [ ]:
if numero_envio_hoje == 4:

  # Aguardar 2 minutos
  time.sleep(120)

  PARCEIROS_MERCADO_LIVELO = ['Carrefour Mercado', 'Carrefour Shopping', "Sam's Club", "Sam's Club - E-commerce", 'Supernosso']
  PARCEIROS_MERCADO_ESFERA = ['Carrefour', 'La Pastina']

  itens_mercado = []

  try:
      for p in extrair_parceiros_livelo():
          if p["parceiro"] in PARCEIROS_MERCADO_LIVELO:
              itens_mercado.append({
                  "parceiro": p["parceiro"],
                  "nome_clube": "🟣 Livelo",
                  "pontuacao": p["pontuacao"],
                  "link_parceiro": p["link_parceiro"],
              })
  except Exception as e:
      print(f"⚠️ Erro ao extrair parceiros de Mercado da Livelo: {e}")

  try:
      for p in extrair_parceiros_esfera():
          if p["parceiro"] in PARCEIROS_MERCADO_ESFERA:
              itens_mercado.append({
                  "parceiro": p["parceiro"],
                  "nome_clube": "🔴 Esfera",
                  "pontuacao": p["pontuacao"],
                  "link_parceiro": p["link_parceiro"],
              })
  except Exception as e:
      print(f"⚠️ Erro ao extrair parceiros de Mercado da Esfera: {e}")

  if itens_mercado:
      # Registra na aba Ranking_dia (+ Ranking_historico), igual as outras
      # categorias, pra também concorrer no ranking do top 5 do envio 7.
      df_mercado = pd.DataFrame([
          {
              "parceiro": item["parceiro"],
              "nome_clube": item["nome_clube"],
              "pontuacao_clube": item["pontuacao"],
              "pontuacao_ajustada": item["pontuacao"],
              "analise_pontuacao": analisar_pontuacao_extracao(item["pontuacao"]),
          }
          for item in itens_mercado
      ])
      registrar_promocoes_ranking(aba_ranking_dia, df_mercado, "Mercado", data_formatada)

      # Criando a mensagem do grupo de Mercado
      msg = ""
      msg_titulo = f"🛒 *Promoções da Categoria de Mercado em {data_formatada}: 👇*"

      for item in itens_mercado:
          msg += textwrap.dedent(f"""
      *Parceiro:* {item["parceiro"]}
      *Programa de Fidelidade:* {item["nome_clube"]}
      *Pontuação:* {formatar_pontuacao_extracao(item["pontuacao"])}
      *Link parceiro:* {item["link_parceiro"]}
      *Análise da Promoção:* {analisar_pontuacao_extracao(item["pontuacao"])}

      {"-"*30}""").strip()

      msg_grupo = msg_titulo + "\n\n" + msg

      payload = {
          "jid": id_grupo_envio,
          "message": msg_grupo,
      }

      response = requests.post(url, json=payload, headers=headers)

      if response.status_code == 200:
          print("Mensagem de Mercado enviada com sucesso para o grupo!")
      else:
          print(f"Erro ao enviar mensagem: {response.status_code} - {response.text}")
          envio_atual_ok = False
  else:
      print("⚠️ Nenhum parceiro de mercado encontrado nesta execução (Livelo/Esfera) - nada enviado.")

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")


## Grupo de Varejo (Envio 5)

### Loja Kabum

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 5:
    df_kabum = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-kabum/",
        parceiro="Kabum",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_kabum)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Ponto

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 5:
    df_ponto = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-ponto/",
        parceiro="Ponto",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_ponto)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Casas Bahia

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 5:
    df_casas_bahia = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-casas-bahia/",
        parceiro="Casas Bahia",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_casas_bahia)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Extra

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 5:
    df_extra = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-extra/",
        parceiro="Extra",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_extra)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Magalu

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 5:
    df_magalu = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-magazine-luiza/",
        parceiro="Magalu",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_magalu)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Mercado Livre

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 5:
    df_mercado_livre = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-mercado-livre/",
        parceiro="Mercado Livre",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_mercado_livre)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja FastShop

In [ ]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 5:
    df_fast_shop = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-fast-shop/",
        parceiro="Fast Shop",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_fast_shop)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Enviando os dados via Whatsapp

In [ ]:
if numero_envio_hoje == 5:

  # Concatenando os dataframes
  df_final = pd.concat([df_kabum, df_mercado_livre, df_magalu, df_ponto, df_casas_bahia, df_extra, df_fast_shop], ignore_index=True)
  df_final

  # Registra as promoções desta categoria na aba Ranking_dia (usado pro
  # ranking do top 5 que sai no envio 7).
  registrar_promocoes_ranking(aba_ranking_dia, df_final, "Varejo", data_formatada)

  # Criando a mensagem do grupo de Beleza e Cosméticos
  msg = ""
  msg_titulo = f"📱💻 *Promoções do Grupo de Varejo em {data_formatada}: 👇*"

  for index, row in df_final.iterrows():
    msg += textwrap.dedent(f"""
  *Parceiro:* {row["parceiro"]}
  *Programa de Fidelidade:* {row["nome_clube"]}
  *Pontuação:* {row["pontuacao_clube"]}
  *Análise da Promoção:* {row["analise_pontuacao"]}

  {"-"*30}""").strip()

  msg_grupo = msg_titulo + "\n""\n" + msg

  # Mensagem
  message_promo = msg_grupo

  # Payload
  payload = {
      "jid": id_grupo_envio, #group_jid,
      "message": message_promo,
  }

  #Enviando as promoções
  response = requests.post(url, json=payload, headers=headers)

  # Resposta
  if response.status_code == 200:
      print("Mensagem com as promoções enviada com sucesso para o grupo!")
  else:
      print(f"Erro ao enviar mensagem: {response.status_code} - {response.text}")
      envio_atual_ok = False

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")


Não é o envio da vez hoje (ou este envio já foi feito).


## Jabares do Envio 3 (Mensagens com Imagens)

### Jabar banco inter (Envio 3, Quarta e Sábado) GRUPO OK

In [ ]:
if numero_envio_hoje == 3 and dia_semana in ("Wednesday", "Saturday"):

  # Aguardar 2 minutos
  time.sleep(120)

  if dia_semana == "Wednesday":
    message_promo = textwrap.dedent("""
    💳 *Quer um cartão de fácil acesso com pontuação e Salas VIPs?*

    O Inter Prime é uma ótima escolha: pontua bem, os pontos não expiram e você ainda
    ganha acesso às Salas VIP de forma gratuita, perfeito para dar um upgrade nas suas viagens.

    Lembrando que a forma mais fácil de conseguir esse cartão é *assinando o plano anual do Duo Gourmet!*

    Acesse o nosso site abaixo para garantir o cupom de desconto no Duo Gormet anual e também o bônus em pontos na abertura da sua conta Inter👇
    🔗 https://murilloborges.com.br/ranking-cartao-de-credito#inter-prime


    -----------------------------

    💳 Não conhece esse cartão? Assista o vídeo abaixo para conhecer os principais benefícios do cartão Inter Prime 👇
    https://www.youtube.com/watch?v=laEm4960klw&list=PLywCMnijM298ruCE-Cey18WvsoV2UqqQf&index=1""").strip()

  else:
    message_promo = textwrap.dedent("""
    *Pensando em começar no mundo dos cartões? 💳*

    O Inter Prime é uma excelente opção: boa pontuação, pontos que não expiram e acesso às Salas VIP ✈️🔥

    A maneira mais simples de conseguir esse cartão é assinando o plano Duo Gourmet anual 👇

    💰 *Quer economizar até R$50 no plano anual do Duo Gourmet?*
    Use o código *B1EE1054* no app do Inter
    👉 https://intergo.app/1f148152

    🟠 *Ainda não tem conta? abra sua conta no Banco Inter e ganhe 200 pontos Loop*
    Use o código *OW1AIMUA*
    👉 https://inter-co.onelink.me/Qyu7/ste2n6tb

    -----------------------------

    💳 Conheça os benefícios do cartão Inter Prime:
    https://www.youtube.com/watch?v=laEm4960klw&list=PLywCMnijM298ruCE-Cey18WvsoV2UqqQf&index=1""").strip()

  drive_id = "1OpOK2Ee1q2yJO-UDhfP3w3hFh2uM7y0s"
  img_url = f"https://drive.google.com/uc?export=view&id={drive_id}"

  payload = {
      "jid": id_grupo_envio, #group_jid,
      "mediaType": "image",
      "mimetype": "image/jpeg",
      "media": img_url,
      "caption": message_promo,  # ← usa seu texto de análise
      "filename": "analise.jpg"
  }

  response = requests.post(url_media, json=payload, headers=headers)

  print(response.json())
  if response.status_code != 200:
      envio_atual_ok = False

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Jabar planilha (Envio 6 na Quinta, Envio 3 no Domingo) GRUPO OK

In [ ]:
if (numero_envio_hoje == 6 and dia_semana == "Thursday") or (numero_envio_hoje == 3 and dia_semana == "Sunday"):

  # Aguardar 2 minutos
  time.sleep(120)

  if dia_semana == "Thursday":
    message_promo = textwrap.dedent("""
    *Quer organizar suas finanças de um jeito simples e inteligente?*

    A nossa Planilha de Controle Financeiro foi feita para quem quer parar de gastar no automático
    e finalmente entender para onde o dinheiro está indo, de forma clara, rápida e sem complicação.

    Por que essa planilha é a solução perfeita para você?
    • Simples de usar e totalmente automatizada
    • Visão completa do seu fluxo de caixa
    • Dashboard visual e intuitivo

    💡 *E o melhor?*
    • Ela custa menos do que uma pizza 🍕
    • Você paga *uma única vez* e fica com a planilha *para sempre*.
    • E você ainda ganha uma aula passo a passo. Mesmo sem saber Excel, você vai conseguir usar sem dificuldade. 🎓✅
    • Ao adquirir, você ainda apoia o nosso trabalho e nos ajuda a continuar trazendo conteúdo gratuito aqui no grupo. 🙏

    👉 Assista ao vídeo e adquira sua planilha aqui:
    https://murilloborges.com.br/lp-planilha-de-controle-financeiro

    -----------------------------

    🧠 *Organização financeira muda tudo.*
    Comece hoje, seu “eu do futuro” vai agradecer.""").strip()
  else:
    message_promo = textwrap.dedent("""
    *Quer organizar suas finanças de um jeito simples?*

    Nossa Planilha de Controle Financeiro te ajuda a ver para onde seu dinheiro está indo e a controlar seus gastos sem complicação.

    • Custa menos que uma pizza 🍕
    • Pagou uma vez, é sua para sempre.
    • E tem aula passo a passo (mesmo se você não souber Excel). 🎓✅

    👉 Conheça e adquira aqui:
    https://murilloborges.com.br/lp-planilha-de-controle-financeiro""").strip()


  # Imagem do Google Drive
  drive_id = "1dHHTOXW8g3bZ4AGZk14Lvnu9mGSpHoVc"
  img_url = f"https://drive.google.com/uc?export=view&id={drive_id}"

  payload = {
      "jid": id_grupo_envio, #group_jid,
      "mediaType": "image",
      "mimetype": "image/jpeg",
      "media": img_url,
      "caption": message_promo,  # ← usa seu texto de análise
      "filename": "analise.jpg"
  }

  response = requests.post(url_media, json=payload, headers=headers)

  print(response.json())
  if response.status_code != 200:
      envio_atual_ok = False

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Jabar TopCashback (Envio 3, Segunda e Quinta)

In [ ]:
if numero_envio_hoje == 3 and dia_semana in ("Monday", "Thursday"):

  #Aguardar 60 segundos
  time.sleep(60)

  if dia_semana == "Monday":

    #Imagem do Google Drive
    drive_id = "1sVCoiFDKu0gbioPQSpZbOVkGbq6NYb8X" #Imagem de Segunda
    img_url = f"https://drive.google.com/uc?export=view&id={drive_id}"

  else:

    #Imagem do Google Drive
    drive_id = "1idhL58jQID5wemKh0XoHfqROn7L3MzTK" #Imagem de Quinta
    img_url = f"https://drive.google.com/uc?export=view&id={drive_id}"

  message_promo = textwrap.dedent("""
  🇬🇧 *Vai viajar ou reservar serviços no exterior? Então presta atenção nisso 👇*

  Alguns parceiros como o ALL por exemplo, permitem reservas no Brasil 🇧🇷

  Se você ainda não tem conta no *TopCashBack do Reino Unido*, vale muito a pena abrir.

  Usando o nosso link, você ainda pode ganhar um *bônus após a primeira compra* 💷✨

  👉 Abra sua conta aqui:
  https://bit.ly/topcashbackreinounido

  -----------------------------

  🏨 *Hospedagens com Cashback em Libra Esterlina:*
  • ALL Accor Live Limitless – até *15% de cashback*
    https://www.topcashback.co.uk/all-accor-live-limitless/

  • Hotels_com – até *15% de cashback*
    https://www.topcashback.co.uk/hotelscom/

  • Expedia – até *8,5% de cashback*
    https://www.topcashback.co.uk/search/merchants/?s=Expedia

  • Agoda – até *4% de cashback*
    https://www.topcashback.co.uk/search/merchants/?s=Agoda

  • Lastminute – até *8,5% de cashback*
    https://www.topcashback.co.uk/search/merchants/?s=lastminute.com

  -----------------------------

  🚗 *Locadoras de Carros com Cashback em Libra Esterlina:*
  • Avis – até *9,35% de cashback*
    https://www.topcashback.co.uk/search/merchants/?s=Avis

  • Discover Cars – até *25,5% de cashback*
    https://www.topcashback.co.uk/discover-car-hire/

  • Europcar – até *11,47% de cashback*
    https://www.topcashback.co.uk/europcar/

  • Sixt UK – até *8,5% de cashback*
    https://www.topcashback.co.uk/sixt-uk/

  • Alamo – até *4,25% de cashback*
    https://www.topcashback.co.uk/alamo/

  -----------------------------

  🚆 *Trem, Ferry e Ônibus com Cashback em Libra Esterlina:*
  • Omio – até *4,25% de cashback*
    https://www.topcashback.co.uk/omio/

  -----------------------------

  🎟️ *Passeios e Atrações com Cashback em Libra Esterlina:*
  • GetYourGuide – até *11% de cashback*
    https://www.topcashback.co.uk/getyourguide/

  • Groupon – até *5,95% de cashback*
    https://www.topcashback.co.uk/groupon/

  • Tripadvisor – até *12% de cashback*
    https://www.topcashback.co.uk/tripadvisor-tours-and-experiences/

  • Disneyland Paris – até *£42.50 de cashback*
    https://www.topcashback.co.uk/disneyland-resort-paris/

  -----------------------------

  ⚠️ *Importante:* sempre leia os termos do cashback antes de comprar.

  ‼️ O percentual pode variar diariamente e cada parceiro possui regras específicas para a validação do retorno.
    """).strip()


  message_promo2 = textwrap.dedent("""
  🇺🇸 *Além do TopCashBack Britânico, também existe o TopCashBack Americano 👇*

  Por isso, a melhor estratégia é sempre consultar os dois sites e comparar qual deles está oferecendo o *maior cashback no momento*.

  Em muitos casos, a diferença pode ser bem relevante 💰

  Alguns parceiros permitem reservas mesmo estando no Brasil 🇧🇷, o que pode gerar cashback em dólar de forma bem interessante 💵

  Se você ainda não tem conta no *TopCashBack dos Estados Unidos*, vale muito a pena abrir.
  Usando o nosso link, você ainda pode ganhar um *bônus após a primeira compra* ✨

  👉 Abra sua conta aqui:
  https://bit.ly/topcashbackamericano

  -----------------------------

  🏨 *Hospedagens*
  • ALL Accor – até *15% de cashback*
    https://www.topcashback.com/accorhotels/

  • Booking – até *10% de cashback*
    https://www.topcashback.com/booking-com/

  • Hotels.com – até *15% de cashback*
    https://www.topcashback.com/hotels-com/

  • Expedia – até *8,5% de cashback*
    https://www.topcashback.com/expedia/

  • Agoda – até *4% de cashback*
    https://www.topcashback.com/agoda/

  -----------------------------

  🚗 *Locadoras de Carros*
  • Avis – até *9,35% de cashback*
    https://www.topcashback.com/avis-rent-a-car/

  • Hertz – até *5% de cashback*
    https://www.topcashback.com/hertz/

  • Europcar – até *9% de cashback*
    https://www.topcashback.com/europcar/

  • Sixt – até *8% de cashback*
    https://www.topcashback.com/sixt/

  • Alamo – até *3% de cashback*
    https://www.topcashback.com/alamo-rent-a-car/

  -----------------------------

  🚆 *Trem, Ferry e Ônibus*
  • Omio – até *10% de cashback*
    https://www.topcashback.com/omio/

  -----------------------------

  🎟️ *Passeios e Atrações*
  • GetYourGuide – até *7% de cashback*
    https://www.topcashback.com/getyourguide/

  • Groupon – até *4% de cashback*
    https://www.topcashback.com/groupon/

  • Viator – até *15% de cashback*
    https://www.topcashback.com/viator/

  • Tripadvisor – até *10% de cashback*
    https://www.topcashback.com/tripadvisor-hotels/

  -----------------------------

  ⚠️ *Importante:* sempre leia os termos do cashback antes de comprar.

  ‼️ O percentual pode variar diariamente e cada parceiro possui regras específicas para a validação do retorno.
  """).strip()


  #Mensagem 1
  payload = {
      "jid": id_grupo_envio,
      "mediaType": "image",
      "mimetype": "image/jpeg",
      "media": img_url,
      "caption": message_promo,
      "filename": "analise.jpg"
  }

  headers = {
      "X-Api-Key": API_KEY,
      "Content-Type": "application/json"
  }

  response = requests.post(url_media, json=payload, headers=headers)
  if response.status_code != 200:
      envio_atual_ok = False

  #Aguardar 20 segundos
  time.sleep(20)


  #Mensagem 2
  message_promo = message_promo2

  payload = {
      "jid": id_grupo_envio,
      "message": message_promo,
  }

  response = requests.post(url, json=payload, headers=headers)
  if response.status_code != 200:
      envio_atual_ok = False

  print("Mensagem enviada!")

  #Aguardar 20 segundos
  time.sleep(20)


  #Mensagem 3
  message_promo3 = textwrap.dedent("""
    📺 *Quer aprender a usar o TopCashBack do jeito certo e extrair o máximo de cashback?*

    No nosso canal do YouTube temos uma *playlist completa* dedicada ao TopCashBack, onde mostramos passo a passo como usar a plataforma, dicas práticas, cuidados importantes e estratégias para aumentar seus ganhos 💰

    É conteúdo ideal tanto para quem está começando quanto para quem já usa e quer melhorar os resultados.

    👉 Assista à playlist completa aqui:
    https://www.youtube.com/playlist?list=PLywCMnijM298Xa4SBJ7w4aQK_qsxgG5nJ
  """).strip()

  message_promo = message_promo3

  payload = {
      "jid": id_grupo_envio,
      "message": message_promo,
  }

  response = requests.post(url, json=payload, headers=headers)
  if response.status_code != 200:
      envio_atual_ok = False

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")



Não é o envio da vez hoje (ou este envio já foi feito).


### Registro do envio 3 (independente do dia)

In [ ]:
if numero_envio_hoje == 3:
  # Registra o resultado do envio 3 (Moda + Jabares do grupo) aqui,
  # fora de qualquer checagem de dia da semana — assim o registro
  # acontece TODO dia que numero_envio_hoje == 3, e não só nos dias
  # em que o Jabar TopCashback (Seg/Qui) também dispara.
  registrar_tentativa_envio(planilha_controle, numero_envio_hoje, "OK" if envio_atual_ok else "Erro")

## Jabar do Envio 5 (Mensagens com Imagens)

### Jabar Visto Americano Hit The Change (5 envio na Segunda) GRUPO OK

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

# Data atual em Brasília
agora = datetime.now(ZoneInfo("America/Sao_Paulo"))

numero_semana = agora.isocalendar().week

if numero_envio_hoje == 5 and dia_semana == "Monday":

    # Aguardar 1 minuto
    time.sleep(60)

    if numero_semana % 2 == 0:
        print("Semana par")

        drive_id = "1WBD7goFmjB1b_ag3j-uGTaa0sLc0_tn-"

        message_promo = textwrap.dedent("""
        🇺🇸 *Vai tirar o visto americano pela primeira vez ou renovar? Não deixe essa etapa ao acaso!*

        Somos parceiros da Hit The Change, uma das principais assessorias especializadas em visto americano, e conseguimos um benefício exclusivo para vocês.

        💰 Pela nossa indicação você ganha R$ 50 de desconto na assessoria para o visto americano.

        Solicitar um visto envolve um investimento importante e estar bem orientado durante todo o processo pode fazer a diferença. Conte com uma equipe especializada para acompanhar você desde a documentação até a entrevista consular.

        📲 Clique no link abaixo para falar com a equipe e garantir seu desconto exclusivo 👇
        https://bit.ly/vistoamericanohtc
        """).strip()

    else:
        print("Semana ímpar")

        drive_id = "1nWonSndcpBmzCmfUmFKrGSBJu83SiHuW"

        message_promo = textwrap.dedent("""
        ✈️ *Seu sonho de conhecer os Estados Unidos começa pelo visto americano*

        Somos parceiros da Hit The Change, uma assessoria especializada que auxilia brasileiros em todo o processo de solicitação do visto americano e renovação. 🇺🇸

        Além de contar com o suporte de uma equipe experiente, quem chega pela nossa indicação garante R$ 50 de desconto na assessoria.

        📲 Clique no link abaixo para conversar com a equipe e aproveitar esse benefício exclusivo.
        https://bit.ly/vistoamericanohtc
        """).strip()

    # Código comum
    img_url = f"https://drive.google.com/uc?export=view&id={drive_id}"

    payload = {
        "jid": id_grupo_envio,
        "mediaType": "image",
        "mimetype": "image/jpeg",
        "media": img_url,
        "caption": message_promo,
        "filename": "analise.jpg"
    }

    headers = {
        "X-Api-Key": API_KEY,
        "Content-Type": "application/json"
    }

    response = requests.post(url_media, json=payload, headers=headers)
    if response.status_code != 200:
        envio_atual_ok = False

    print("Mensagem enviada!")

### Registro do envio 5 (independente do dia)

In [ ]:
if numero_envio_hoje == 5:
  # Mesma lógica: registra o envio 5 (Varejo + Jabar Visto) todo dia
  # que numero_envio_hoje == 5, não só quando o Jabar Visto (Segunda)
  # também dispara.
  registrar_tentativa_envio(planilha_controle, numero_envio_hoje, "OK" if envio_atual_ok else "Erro")

## Jabares do Envio 7 (Mensagens com Imagens)

### Jabar cartão Ruby Stell (Envio 7, Terça-Feira) — DESATIVADO TEMPORARIAMENTE

In [ ]:
# DESATIVADO TEMPORARIAMENTE em 2026-09 (colidia com o Get Your Guide na terca).
# Pra reativar, so remover o "and False" da condicao abaixo.
if numero_envio_hoje == 7 and dia_semana in ("Tuesday") and False:

  # Aguardar 1 minuto
  time.sleep(60)

  if dia_semana == "Friday":

    # Imagem do Google Drive
    drive_id = "1EV2mu8AfLkIzo9dM23nP58e5lN7bYN18" #Imagem de Sexta
    img_url = f"https://drive.google.com/uc?export=view&id={drive_id}"

    message_promo = textwrap.dedent("""
    *Vai viajar para fora do Brasil ou fazer uma compra internacional? Então presta atenção nisso 👇*

    O *Cartão Crypto Ruby* é simplesmente um dos melhores cartões para uso internacional, porque:
    • Não cobra IOF nas compras (3,5% de economia) 🌍
    • Tem spread super baixo 💱
    • *2% de cashback* em todas as compras 💸
    • Spotify grátis 🎧

    Ou seja: você gasta menos, economiza nas taxas e ainda recebe dinheiro de volta.

    -----------------------------

    💳 *Quer ver esse cartão funcionando na prática?*
    Eu testei esse cartão por 15 dias e gravei um vídeo mostrando como usar, benefícios e pontos de atenção 👇
    https://www.youtube.com/watch?v=OA7LlqiobHQ&t=300s

    -----------------------------

    🔵 Gostou do cartão?
    Abra sua conta na Crypto com o nosso link e ganhe até **$60 USD** de bônus 💵
    👉 https://bit.ly/wcryptoruby

    """).strip()

  else:

    # Imagem do Google Drive
    drive_id = "1eLpOE5cJJe2ahFIMbbY1do2RSE-adggr" #Imagem de Terça
    img_url = f"https://drive.google.com/uc?export=view&id={drive_id}"

    message_promo = textwrap.dedent("""
    *Se você vai viajar ou costuma comprar fora do Brasil, esse cartão pode te fazer economizar MUITO* ✈️💸

    O *Crypto Ruby* é um dos poucos cartões que realmente vale a pena no uso internacional:

    • Zero IOF nas compras (economia 3,5%) 🌍
    • Spread super baixo (conversão mais justa) 💱
    • *2% de cashback* em tudo que você gastar 💸
    • Spotify grátis 🎧

    👀 *Spoiler:* ele tem um custo muito menor que vários cartões globais famosos do mercado.

    -----------------------------

    🎥 *Quer ver como ele funciona na prática?*
    Eu usei esse cartão e gravei um vídeo mostrando tudo:
    https://www.youtube.com/watch?v=OA7LlqiobHQ&t=300s

    -----------------------------

    🔵 Curtiu a ideia?
    Abra sua conta pela Crypto e ganhe até *$60 USD* de bônus 💵
    👉 https://bit.ly/wcryptoruby
    """).strip()

  payload = {
      "jid": id_grupo_envio, #group_jid,
      "mediaType": "image",
      "mimetype": "image/jpeg",
      "media": img_url,
      "caption": message_promo,
      "filename": "analise.jpg"
  }

  headers = {
      "X-Api-Key": API_KEY,
      "Content-Type": "application/json"
  }

  response = requests.post(url_media, json=payload, headers=headers)
  if response.status_code != 200:
      envio_atual_ok = False

  print("Mensagem enviada!")

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Jabar Passeios Get Your Guide (Envio 7, Terça-Feira e Sexta-Feira)

In [ ]:
if numero_envio_hoje == 7 and dia_semana in ("Tuesday", "Friday"):

    # Aguardar 5 segundos
    time.sleep(5)

    drive_id = "1F3Hq6nsn4fzS5KwZy4Addf65sY1AW7AG"
    img_url = f"https://drive.google.com/uc?export=view&id={drive_id}"

    if dia_semana == "Monday":
        message_promo = textwrap.dedent("""
        🌍 *Quer transformar sua viagem em uma experiência inesquecível?*

        No Get Your Guide você encontra os melhores passeios, tours e experiências em destinos do mundo todo — tudo em um só lugar! 🗺️

        🎯 City tours, passeios de barco, experiências gastronômicas, trilhas, museus e muito mais, em centenas de destinos pelo mundo.

        👉 *Acesse nossas recomendações e escolha o passeio perfeito para a sua próxima viagem:*
        https://www.getyourguide.com/explorer/creators/-cs4294

        💰 *Primeira reserva? Use o cupom abaixo e ganhe 5% de desconto:*
        🎟️ Cupom: *MURILLOINVESTOR5*

        -----------------------------

        ✈️ Viagem planejada é viagem aproveitada. Garante já o seu passeio!
        """).strip()

    else:  # Thursday
        message_promo = textwrap.dedent("""
        ✈️ *Já tem viagem marcada? Então esse conteúdo é pra ti!*

        No Get Your Guide tu encontras passeios incríveis em destinos do mundo todo — desde city tours clássicos até experiências únicas que vão fazer a tua viagem ser ainda mais especial 🌍🔥

        🗺️ Temos opções separadas por país pra facilitar a tua busca!

        👉 *Dá uma olhada nas nossas recomendações:*
        https://www.getyourguide.com/explorer/creators/-cs4294

        💰 *Na tua primeira reserva, usa o cupom abaixo e garante 5% de desconto:*
        🎟️ *MURILLOINVESTOR5*

        -----------------------------

        🌟 Experiências boas não acontecem por acaso — elas são planejadas!
        """).strip()

    payload = {
        "jid": id_grupo_envio,
        "mediaType": "image",
        "mimetype": "image/jpeg",
        "media": img_url,
        "caption": message_promo,
        "filename": "getyourguide.jpg"
    }

    response = requests.post(url_media, json=payload, headers=headers)
    if response.status_code != 200:
        envio_atual_ok = False

    print("Mensagem enviada!")

else:
    print("Não é o envio da vez hoje, ou hoje não é dia de Get Your Guide.")

Não é o envio da vez hoje, ou hoje não é dia de Get Your Guide.


## Grupo de Farmácia e Medicamentos (Envio 7)

In [ ]:
if numero_envio_hoje == 7:

  # Aguardar 1 minuto
  time.sleep(60)

  # Parceiros de farmácia/medicamentos já identificados no catálogo da
  # Livelo e da Esfera (ver extrair_parceiros_livelo/extrair_parceiros_esfera
  # lá na célula "Extração Livelo e Esfera"). Essa categoria NÃO usa o
  # comparemania.com.br - usa direto o catálogo oficial de cada programa.
  PARCEIROS_FARMACIA_LIVELO = ["Farmacias App", "Drogaria Sao Paulo", "Drogarias Pacheco", "Drogal"]
  PARCEIROS_FARMACIA_ESFERA = ["Drogal", "Drogaria Araujo", "Drogaria Venancio", "Extrafarma", "Pague Menos"]

  def analisar_pontuacao_farmacia(pontuacao):
      # Mesma escala de análise usada em coletar_promocoes() lá em cima -
      # Livelo/Esfera não dividem por 1,3 (a divisão é só pra normalizar
      # OUTROS clubes contra Livelo/Esfera; aqui já é o valor direto deles).
      if pontuacao >= 10:
          return '⭐⭐⭐⭐⭐ Nível 5 (Excelente / Raro) 😏: Promoção rara, aproveite sem medo!'
      elif pontuacao >= 8:
          return '⭐⭐⭐⭐ Nível 4 (Muito Bom) 😎: Ótima Promoção, daqui pra cima já vale muito!'
      elif pontuacao >= 5:
          return '⭐⭐⭐ Nível 3 (Bom) 😉: Bom momento para potencializar compras planejadas, mas se puder aguardar, tem coisa melhor!'
      elif pontuacao >= 3:
          return '⭐⭐ Nível 2 (Mediano) 🧐: Dá pra usar se você realmente já iria comprar, mas é melhor aguardar algo melhor ;)'
      else:
          return '⭐ Nível 1 (Ruim) 😡: Sugiro aguardar algo melhor'

  def formatar_pontuacao_farmacia(valor):
      return str(int(valor)) if valor == int(valor) else str(round(valor, 2))

  itens_farmacia = []

  try:
      for p in extrair_parceiros_livelo():
          if p["parceiro"] in PARCEIROS_FARMACIA_LIVELO:
              itens_farmacia.append({
                  "parceiro": p["parceiro"],
                  "nome_clube": "🟣 Livelo",
                  "pontuacao": p["pontuacao"],
                  "link_parceiro": p["link_parceiro"],
              })
  except Exception as e:
      print(f"⚠️ Erro ao extrair parceiros de farmácia da Livelo: {e}")

  try:
      for p in extrair_parceiros_esfera():
          if p["parceiro"] in PARCEIROS_FARMACIA_ESFERA:
              itens_farmacia.append({
                  "parceiro": p["parceiro"],
                  "nome_clube": "🔴 Esfera",
                  "pontuacao": p["pontuacao"],
                  "link_parceiro": p["link_parceiro"],
              })
  except Exception as e:
      print(f"⚠️ Erro ao extrair parceiros de farmácia da Esfera: {e}")

  if itens_farmacia:
      # Registra na aba Ranking_dia (+ Ranking_historico), igual as outras
      # categorias, pra também concorrer no ranking do top 5 do envio 7.
      df_farmacia = pd.DataFrame([
          {
              "parceiro": item["parceiro"],
              "nome_clube": item["nome_clube"],
              "pontuacao_clube": item["pontuacao"],
              "pontuacao_ajustada": item["pontuacao"],
              "analise_pontuacao": analisar_pontuacao_farmacia(item["pontuacao"]),
          }
          for item in itens_farmacia
      ])
      registrar_promocoes_ranking(aba_ranking_dia, df_farmacia, "Farmácia e Medicamentos", data_formatada)

      # Criando a mensagem do grupo de Farmácia e Medicamentos
      msg = ""
      msg_titulo = f"💊🩺 *Promoções da Categoria de Farmácia e Medicamentos em {data_formatada}: 👇*"

      for item in itens_farmacia:
          msg += textwrap.dedent(f"""
      *Parceiro:* {item["parceiro"]}
      *Programa de Fidelidade:* {item["nome_clube"]}
      *Pontuação:* {formatar_pontuacao_farmacia(item["pontuacao"])}
      *Link parceiro:* {item["link_parceiro"]}
      *Análise da Promoção:* {analisar_pontuacao_farmacia(item["pontuacao"])}

      {"-"*30}""").strip()

      msg_grupo = msg_titulo + "\n\n" + msg

      payload = {
          "jid": id_grupo_envio,
          "message": msg_grupo,
      }

      response = requests.post(url, json=payload, headers=headers)

      if response.status_code == 200:
          print("Mensagem de Farmácia e Medicamentos enviada com sucesso para o grupo!")
      else:
          print(f"Erro ao enviar mensagem: {response.status_code} - {response.text}")
          envio_atual_ok = False
  else:
      print("⚠️ Nenhum parceiro de farmácia encontrado nesta execução (Livelo/Esfera) - nada enviado.")

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")


### Registro do envio 7 (independente do dia)

In [ ]:
if numero_envio_hoje == 7:
  # Mesma lógica: registra o envio 7 (Jabares Noturnos) todo dia que
  # numero_envio_hoje == 7, não só Terça/Sexta (Ruby Stell) ou
  # Terça/Sexta (Get Your Guide).
  registrar_tentativa_envio(planilha_controle, numero_envio_hoje, "OK" if envio_atual_ok else "Erro")

### Ranking do dia - Top 5 melhores promoções (envio 7)

In [ ]:
if numero_envio_hoje == 7:

  # =========================================================================
  # 🏆 RANKING DO DIA — TOP 5 MELHORES PROMOÇÕES
  # =========================================================================
  # Lê a aba "Ranking_dia" (zerada hoje de manhã no envio 1 e preenchida
  # pelos envios 1 a 5 com a pontuação de cada promoção) e manda um
  # resumo com as 5 melhores do dia, ordenadas da maior pontuação pra
  # menor. Roda em TODOS os dias, sem checagem de dia da semana.

  linhas_ranking = aba_ranking_dia.get_all_values()[1:]  # pula o cabeçalho

  # Monta uma lista de dicts e converte pontuacao_ajustada pra float, pra
  # poder ordenar numericamente (na planilha tudo vem como texto).
  promocoes_do_dia = []
  for row in linhas_ranking:
      if len(row) < 7:
          continue
      try:
          pontuacao_ajustada = float(row[5].replace(",", "."))
      except ValueError:
          continue
      promocoes_do_dia.append({
          "categoria": row[1],
          "parceiro": row[2],
          "nome_clube": row[3],
          "pontuacao_clube": row[4],
          "pontuacao_ajustada": pontuacao_ajustada,
          "analise_pontuacao": row[6],
      })

  promocoes_do_dia.sort(key=lambda p: p["pontuacao_ajustada"], reverse=True)
  top5 = promocoes_do_dia[:5]

  if top5:
      medalhas = ["🥇", "🥈", "🥉", "4️⃣", "5️⃣"]
      msg_ranking = f"🏆 *Resumo do dia: TOP {len(top5)} melhores promoções* 👇\n\n"
      for posicao, promo in enumerate(top5):
          msg_ranking += textwrap.dedent(f"""
          {medalhas[posicao]} *{promo['nome_clube']}* ({promo['categoria']})
          Parceiro: {promo['parceiro']}
          Pontuação: {promo['pontuacao_clube']} pts
          {promo['analise_pontuacao']}
          """).strip() + "\n\n"
      msg_ranking = msg_ranking.strip()

      payload_ranking = {
          "jid": id_grupo_envio,
          "message": msg_ranking,
      }
      response_ranking = requests.post(url, json=payload_ranking, headers=headers, timeout=30)
      if response_ranking.status_code == 200:
          print("🏆 Ranking do dia enviado com sucesso para o grupo!")
      else:
          print(f"⚠️ Erro ao enviar ranking do dia: {response_ranking.status_code} - {response_ranking.text}")
  else:
      print("ℹ️ Nenhuma promoção registrada hoje na aba Ranking_dia — ranking não enviado.")

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")

## Transferências Bonificadas

In [ ]:
if numero_envio_hoje == 6:

  import requests
  from bs4 import BeautifulSoup
  from google import genai
  from google.genai import types
  import re
  import datetime
  from zoneinfo import ZoneInfo
  import os

  # =========================================
  # 🔑 CHAVE DA API DA IA (Gemini)
  # =========================================
  client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

  # =========================================
  # 🔐 GOOGLE SHEETS
  # =========================================
  # A conexão (spreadsheet), feriados e eventos já foram carregados uma
  # única vez lá no topo do notebook, na célula "Controle de Envios".
  # Aqui só recarregamos feriados/eventos para garantir dados atualizados.
  feriados = carregar_feriados(spreadsheet)
  eventos = carregar_eventos(spreadsheet)
  print(f"✅ {len(feriados)} feriados e {len(eventos)} eventos carregados.")

  # =========================================
  # 🔎 EXTRAÇÃO DE LINKS DE PROMOÇÕES
  # =========================================
  def extrair_links_promocoes():

      # 🐙 VERSÃO GITHUB ACTIONS (ativa)
      url = os.environ["URL_SCRAPING"]
      url_two = os.environ["URL_SCRAPING_TWO"]

      # 🖥️ VERSÃO LOCAL (Colab)
      #url = ler_config_local("url_scraping.txt")
      #url_two = ler_config_local("url_scraping_two.txt")

      headers = {"User-Agent": "Mozilla/5.0"}
      response = requests.get(url, headers=headers)
      soup = BeautifulSoup(response.text, "html.parser")

      links = []
      for a in soup.find_all("a", href=True):
          href = a["href"]
          if (
              url_two in href
              and "pontos" in href
              and not any(x in href for x in ["categoria", "tag", "page", "#"])
          ):
              links.append(href)

      # dict.fromkeys() remove duplicatas preservando a ORDEM de inserção
      # (diferente de set(), que embaralha). O site lista as matérias mais
      # recentes primeiro no HTML, então isso garante processar/enviar da
      # mais nova pra mais antiga, em vez de ordem aleatória por hash.
      return list(dict.fromkeys(links))

  def carregar_status_links(spreadsheet):
      """Lê a aba 'Status Links' e retorna um dict {link: status}, sempre
      com o registro mais recente de cada link. Usado para pular só os
      links já CONFIRMADOS como vencidos — os demais (novos ou ainda
      válidos) continuam sendo reprocessados, pra permitir reenviar como
      lembrete enquanto a promoção durar."""
      aba = spreadsheet.worksheet("Status Links")
      dados = aba.get_all_values()
      status_por_link = {}
      for row in dados[1:]:  # pula o cabeçalho
          if len(row) >= 2 and row[0]:
              link = row[0].strip()
              status = row[1].strip()
              status_por_link[link] = status  # linhas mais abaixo sobrescrevem (mais recentes)
      return status_por_link

  def registrar_status_link(spreadsheet, link, status, data_validade):
      """Registra o status mais recente de um link (Dentro do prazo / Fora
      do prazo) na aba 'Status Links' — usado nas próximas execuções pra
      decidir se ele deve ser pulado (só quando vencido) ou reprocessado."""
      aba = spreadsheet.worksheet("Status Links")
      agora_str = agora_brasilia.strftime("%d/%m/%Y %H:%M:%S")
      linha = [link, status, data_validade or "", agora_str]
      aba.append_row(linha, value_input_option="USER_ENTERED")

  # =========================================
  # 🤖 EXTRAÇÃO COM IA
  # =========================================
  def extrair_com_ia(html_texto, data_publicacao):
    from datetime import datetime
    from zoneinfo import ZoneInfo
    hoje = datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%d/%m/%Y")

    prompt = f"""
    Você é um especialista em milhas aéreas e vai extrair dados importantes e montar uma mensagem que será enviada em um grupo.
    Seja preciso e nunca invente informações que não estejam explícitas no texto.

    ─────────────────────────────────────
    PARCEIRO (banco de origem)
    ─────────────────────────────────────
    - O parceiro é sempre o DE (origem dos pontos) e a companhia aérea/programa é o PARA (destino das milhas).
      Exemplo: ESFERA → GOL (Smiles)
    - Se a promoção tiver apenas 1 parceiro identificável, use o nome dele conforme as regras abaixo.
    - Se tiver exatamente 2 parceiros, cite os dois nomes. Exemplo: ESFERA e Livelo
    - Se tiver 3 ou mais parceiros, use "Vários parceiros".
    - Nunca invente um parceiro se não estiver explícito no texto.
    - Nunca confunda banco com companhia aérea.

    ─────────────────────────────────────
    NOMES PADRONIZADOS
    ─────────────────────────────────────
    - Azul / Tudo Azul → AZUL (Tudo Azul)
    - GOL / Smiles → GOL (Smiles)
    - Latam / Latam Pass → Latam (Latam Pass)
    - Iberia → Iberia Club
    - Livelo → Livelo
    - Qualquer outro → primeira letra maiúscula, restante minúsculo

    ─────────────────────────────────────
    ÍCONES POR COMPANHIA/PROGRAMA DE DESTINO
    ─────────────────────────────────────
    - AZUL (Tudo Azul) → 🔵
    - Latam (Latam Pass) → 🔵
    - ALL Accor → 🔵
    - KrisFlyer → 🔵
    - GOL (Smiles) → 🟠
    - Iberia Club → 🔴
    - Sicredi → 🟢
    - Banco do Brasil / BB → 🟡
    - C6 Bank → ⚫️
    - Qualquer outro → ⚪️

    ─────────────────────────────────────
    DATA DE VALIDADE
    ─────────────────────────────────────
    - Esta matéria foi publicada em {data_publicacao}. Se o texto mencionar a
      validade citando SOMENTE o dia (sem mês explícito), use o mês e o ano
      da PUBLICAÇÃO da matéria acima — NUNCA use o mês/ano de hoje pra isso,
      porque esta matéria pode estar sendo reprocessada dias ou semanas
      depois de ter sido publicada.
    - A data de hoje é {hoje}. Use isso só como contexto geral, nunca para
      inferir o mês/ano de uma data escrita parcialmente no texto.
    - A promoção é válida até às 23h59 do dia informado.
    - Não confunda a data de publicação do artigo com a data de validade da promoção.
      A data de validade geralmente aparece como: "promoção se encerra em", "válida até", "transferências até".
    - Se não encontrar nenhuma data de validade no texto, escreva: 📅 Data não informada

    ─────────────────────────────────────
    COMPANHIA AÉREA vs PROGRAMA DE HOTEL
    ─────────────────────────────────────
    - Mostre ✈️ Companhia aérea: somente quando o destino for uma cia aérea.
    - Mostre 🏨 Programa de fidelidade de hotel: somente quando o destino for ALL Accor.
    - NUNCA mostre as duas linhas na mesma mensagem — é sempre uma ou outra.
    - O parceiro NUNCA é Azul/Tudo Azul quando o destino é ALL Accor. Sempre o contrário: parceiro ALL Accor → destino AZUL (Tudo Azul).

    ─────────────────────────────────────
    EXEMPLO PRÁTICO
    ─────────────────────────────────────
    - Padrão fixo: Enviou 10.000 pontos.
    - Calcule o Recebeu com base no bônus máximo informado.
      Exemplo: 100% de bônus → Enviou 10.000 pontos → Recebeu 20.000 milhas.
    - Use ponto como separador de milhar (padrão brasileiro): 20.000, não 20,000.
    - Sempre indique o percentual considerado: "Considerando X% de Bônus".

    ─────────────────────────────────────
    ALERTA FINAL
    ─────────────────────────────────────
    - Sempre inclua o aviso para ler o regulamento antes de participar.

    ─────────────────────────────────────
    TEXTO DA PROMOÇÃO PARA ANALISAR:
    ─────────────────────────────────────
    {html_texto}

    ─────────────────────────────────────
    FORMATO DE SAÍDA (siga exatamente):
    ─────────────────────────────────────

    🔵 Alerta de Transferência Bonificada: C6 BANK para AZUL (Tudo Azul) 🔵
    ━━━━━━━━━━

    🚀 Transfira seus pontos do C6 BANK para o programa Tudo Azul e ganhe milhas bônus!

    🏦 Parceiro: C6 BANK
    ✈️ Companhia aérea: AZUL (Tudo Azul)  [somente quando for cia aérea]
    🏨 Programa de fidelidade de hotel: ALL Accor  [somente quando o destino for ALL Accor]

    🔥 BONIFICAÇÃO: Até 130% de bônus (Consulte as regras para o seu percentual específico)

    📱 EXEMPLO PRÁTICO:
    Enviou: 10.000 pontos C6 BANK
    Recebeu: 23.000 milhas na Azul (Considerando 130% de Bônus)

    📅 Válido até 19/03/2026

    ⚠️ Atenção: Leia o regulamento da promoção antes de participar!
    """

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction="Extraia dados de promoções de milhas.",
            temperature=0,
        ),
    )

    return response.text

  # =========================================
  # 🗓️ FUNÇÕES AUXILIARES DE DATA
  # =========================================
  def nome_dia_semana_pt(data):
      dias = {0: "Segunda-feira", 1: "Terça-feira", 2: "Quarta-feira", 3: "Quinta-feira",
              4: "Sexta-feira", 5: "Sábado", 6: "Domingo"}
      return dias[data.weekday()]

  def nome_mes_pt(data):
      meses = {1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
              5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
              9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"}
      return meses[data.month]

  # =========================================
  # ▶️ EXECUÇÃO PRINCIPAL
  # =========================================
  list_promocao_key = []
  #print(f'Imprimindo a lista na primeira execucao: {list_promocao_key}')

  if __name__ == "__main__":

      links = extrair_links_promocoes()

      # ── Só pula links já CONFIRMADOS como vencidos. Os demais (novos ou
      # ainda válidos numa checagem anterior) continuam sendo reprocessados,
      # pra permitir reenviar a promoção como lembrete enquanto durar. ──
      status_links = carregar_status_links(spreadsheet)
      links_nao_vencidos = [l for l in links if status_links.get(l) != "Fora do prazo"]

      # ── Limita quantas matérias processar por execução ──
      # A lista já vem da mais recente pra mais antiga, então isso sempre
      # prioriza as promoções mais novas primeiro.
      MAX_LINKS_POR_EXECUCAO = 15
      links_para_processar = links_nao_vencidos[:MAX_LINKS_POR_EXECUCAO]

      print(f"📋 {len(links)} links encontrados | {len(links_nao_vencidos)} não vencidos | processando até {len(links_para_processar)} nesta execução.")

      for i, link in enumerate(links_para_processar, start=1):
          print(f"\n{'='*50}")
          #print(f"{i}. {link}")

          headers = {"User-Agent": "Mozilla/5.0"}
          response = requests.get(link, headers=headers)
          soup = BeautifulSoup(response.text, "html.parser")
          html_texto = soup.get_text(separator=" ", strip=True)

          # ── Data real de publicação da matéria (extraída do HTML, não da IA) ──
          meta_publicado = soup.find("meta", attrs={"property": "article:published_time"})
          if meta_publicado and meta_publicado.get("content"):
              data_publicacao_dt = datetime.datetime.fromisoformat(meta_publicado["content"]).astimezone(ZoneInfo("America/Sao_Paulo"))
              data_publicacao_str = data_publicacao_dt.strftime("%d/%m/%Y")
          else:
              data_publicacao_dt = None
              data_publicacao_str = "Data de publicação não encontrada"

          print(f"Publicada em: {data_publicacao_str}")

          resultado = extrair_com_ia(html_texto, data_publicacao_str)
          msg_link_grupo = f"🔗 Mais informações sobre a promoção: {link}"
          resultado += f"\n\n{msg_link_grupo}\n\n"

          # ── Extrai data de validade ──
          match = re.search(r"\d{2}/\d{2}/\d{4}", resultado)

          if match:
              data_str = match.group()
              # Nome diferente de "data_formatada" DE PROPÓSITO — essa é a
              # data de VALIDADE da promoção (varia por matéria), enquanto
              # "data_formatada" é uma variável GLOBAL do notebook inteiro
              # (a data de hoje, usada em registrar_tentativa_envio,
              # gravar_no_sheets etc). Usar o mesmo nome aqui sobrescrevia
              # a global com um objeto datetime, quebrando tudo que
              # rodasse depois deste loop na mesma execução.
              data_validade_dt = datetime.datetime.strptime(data_str, "%d/%m/%Y")
              data_promocao = data_validade_dt.date()
              data_atual = datetime.datetime.now(ZoneInfo("America/Sao_Paulo")).date()

              # ── Trava de segurança: validade não pode ser antes da publicação ──
              if data_publicacao_dt and data_promocao < data_publicacao_dt.date():
                  status_promo = "Fora do prazo"
                  print(f"⚠️ Data de validade ({data_str}) é anterior à publicação ({data_publicacao_str}) — descartando.")
              else:
                  status_promo = "Dentro do prazo" if data_promocao >= data_atual else "Fora do prazo"
          else:
              data_str = None
              data_promocao = None
              status_promo = "Fora do prazo"

          print(f"Status: {status_promo}")

          # ── Registra o status atual do link, pra decidir nas próximas
          # execuções se ele deve ser pulado (só quando vencido) ou não ──
          registrar_status_link(spreadsheet, link, status_promo, data_str)

          # ── Alerta de último dia — lógica no Python, não na IA ──
          if data_promocao and data_promocao == data_atual:
            alerta_hoje = "\n\n🚨 ATENÇÃO: Essa promoção acaba HOJE! Corra para aproveitar!\n"
            resultado = re.sub(
                r"(📅 Váli?do? até \d{2}/\d{2}/\d{4})",
                r"\1" + alerta_hoje,
                resultado
            )

          # ── Extrai campos da mensagem ──
          parceiro = re.search(r"🏦 Parceiros?:\s*(.+)", resultado)
          parceiro = parceiro.group(1).strip() if parceiro else None

          cia = re.search(r"✈️ Companhia aérea:\s*(.+)", resultado)
          cia = cia.group(1).strip() if cia else None

          if not cia:
              hotel = re.search(r"🏨 Programa de fidelidade de hotel:\s*(.+)", resultado)
              cia = hotel.group(1).strip() if hotel else None

          bonus = re.search(r"BONIFICAÇÃO:.*?(\d+)%", resultado)
          bonus_str = bonus.group(1) + "%" if bonus else None
          bonus_num = int(bonus.group(1)) if bonus else None

          # ── Link Clube Azul — lógica no Python, não na IA ──
          LINK_CLUBE_AZUL = "https://apps.voeazul.com.br/TudoAzulClub/share-mgm.html?hash=7uhPhMsj%2B2YY%2Bczpr2ebrxpa6feTjo%2F5yu2MJlXj4ysc-aC"
          TEXTO_CLUBE_AZUL = f"\n💰 Assine o Clube Azul com o nosso link para ganhar a bonificação maior + 1.000 milhas bônus (a gente também ganha esse bônus): {LINK_CLUBE_AZUL}\n"

          LINK_CLUBE_SMILES = "https://www.smiles.com.br/indicar-amigos/indicado?icode=1-123690609266"
          CODIGO_INDICACAO_SMILES = '1-123690609266'
          TEXTO_CLUBE_SMILES = f"\n💰 Assine o Clube Smiles com o nosso link para ganhar a bonificação maior + 1.000 milhas bônus (a gente também ganha esse bônus): {LINK_CLUBE_SMILES}\n Obs: Se o código não for aplicado automaticamente, adicione o código {CODIGO_INDICACAO_SMILES} na adesão do clube 🚀\n\n"

          if cia and "azul" in cia.lower():
              resultado = resultado.replace(
                  "🔥 BONIFICAÇÃO:",
                  TEXTO_CLUBE_AZUL + "🔥 BONIFICAÇÃO:"
              )
          elif cia and "smiles" in cia.lower():
              resultado = resultado.replace(
                  "🔥 BONIFICAÇÃO:",
                  TEXTO_CLUBE_SMILES + "🔥 BONIFICAÇÃO:"
              )

          # ── Monta a chave anti-duplicata (dentro da mesma execução) ──
          parceiro_key = parceiro or ""
          cia_key = cia or ""
          data_key = data_str or ""
          promocao_key = (parceiro_key + cia_key + data_key).replace(" ", "")

          cont = 0
          if promocao_key not in list_promocao_key and status_promo == "Dentro do prazo":
              list_promocao_key.append(promocao_key)

              # id_grupo_envio, INSTANCE_ID e API_KEY já foram definidos lá no topo
              # do notebook (respeitando a flag ENVIAR_PARA_GRUPO_TESTE) — não
              # precisa redefinir aqui, e assim essa mensagem também respeita a flag.

              url_ws = f"https://api.zapperapi.com/{INSTANCE_ID}/messages/text"
              headers_ws = {
                  "X-Api-Key": API_KEY,
                  "Content-Type": "application/json"
              }
              payload = {
                  "jid": id_grupo_envio,
                  "message": resultado,
              }

              # Aguardar 20 segundos
              time.sleep(20)
              response_ws = requests.post(url_ws, json=payload, headers=headers_ws)
              print("📨 Promoção enviada no grupo!")

              # 🔔 Alerta pessoal: toda transferência bonificada nova enviada
              enviar_alerta_pessoal(
                  "Transferência bonificada",
                  f"{parceiro or 'Parceiro'} → {cia or 'programa'}"
                  + (f": {bonus_str} de bônus" if bonus_str else "")
                  + (f"\nVálido até {data_str}" if data_str else "")
                  + f"\n{link}"
              )

              # ── Monta ID da promoção ──
              parceiro_slug = (parceiro or "").lower().replace(" ", "-")
              cia_slug = (cia or "").lower().replace(" ", "-")
              bonus_slug = str(bonus_num) if bonus_num else "0"
              data_slug = f"Valido ate {data_str}" if data_str else "sem-data"
              id_promocao = f"{parceiro_slug}-{cia_slug}-{bonus_slug}-{data_slug}"

              agora = datetime.datetime.now(ZoneInfo("America/Sao_Paulo"))
              data_hora_str = agora.strftime("%d/%m/%Y %H:%M:%S")
              data_hoje = agora.date()
              nome_dia = nome_dia_semana_pt(agora)
              nome_mes = nome_mes_pt(agora)

              flag_feriado = 1 if data_hoje in feriados else 0
              nome_feriado = feriados.get(data_hoje, "")

              flag_evento = 1 if data_hoje in eventos else 0
              nome_evento = eventos.get(data_hoje, "")

              texto_original = html_texto[:500].replace("\n", " ").strip()

              linha = [
                  data_hora_str,
                  agora.strftime("%d/%m/%Y"),
                  nome_dia,
                  nome_mes,
                  id_promocao,
                  parceiro or "",
                  cia or "",
                  bonus_str or "",
                  bonus_num or 0,
                  data_str or "",
                  flag_feriado,
                  nome_feriado,
                  flag_evento,
                  nome_evento,
                  texto_original,
                  link
              ]

              gravar_no_sheets(spreadsheet, linha)
          else:
              print("⚠️  Promoção duplicada e/ou fora do prazo — ignorada.")

      # ── Chegar até aqui sem exceção = execução considerada bem-sucedida.
      # Falhas de envio individuais (por link) já são tratadas e logadas
      # acima sem interromper o loop; uma falha fatal (ex: erro de conexão
      # não tratado) impediria a execução de chegar nesta linha, e nesse
      # caso nenhuma linha é registrada aqui — a próxima execução tenta de
      # novo naturalmente. ──
      registrar_tentativa_envio(planilha_controle, numero_envio_hoje, "OK")

✅ 36 feriados e 54 eventos carregados.
📋 31 links encontrados | 31 não vencidos | processando até 15 nesta execução.

Publicada em: 11/07/2026
Status: Dentro do prazo
📨 Promoção enviada no grupo!
✅ Linha gravada no Google Sheets!

Publicada em: 10/07/2026
Status: Fora do prazo
⚠️  Promoção duplicada e/ou fora do prazo — ignorada.

Publicada em: 09/07/2026
Status: Fora do prazo
⚠️  Promoção duplicada e/ou fora do prazo — ignorada.

Publicada em: 08/07/2026
Status: Fora do prazo
⚠️  Promoção duplicada e/ou fora do prazo — ignorada.

Publicada em: 08/07/2026
Status: Fora do prazo
⚠️  Promoção duplicada e/ou fora do prazo — ignorada.

Publicada em: 07/07/2026
Status: Fora do prazo
⚠️  Promoção duplicada e/ou fora do prazo — ignorada.

Publicada em: 07/07/2026
Status: Fora do prazo
⚠️  Promoção duplicada e/ou fora do prazo — ignorada.

Publicada em: 07/07/2026
Status: Fora do prazo
⚠️  Promoção duplicada e/ou fora do prazo — ignorada.

Publicada em: 06/07/2026
Status: Fora do prazo
⚠️  Prom

## Captura Geral de Promoções (fila de aprovação)

In [ ]:
# =========================================================================
# 🆕 CAPTURA E FILA DE APROVAÇÃO DE PROMOÇÕES GERAIS
# =========================================================================
# Setup (conexões, funções) e os Passos 1+2 (processar aprovações/recusas e
# enviar as aprovadas) rodam em TODA execução do notebook — assim, uma
# promoção marcada "Aprovado" sai o mais rápido possível, sem esperar até
# 1 ou 4 chegar de novo. Só o Passo 3 (raspar as 10 fontes por promoções
# novas) fica restrito a numero_envio_hoje 1 e 4, pra não gastar cota
# desnecessária raspando toda hora.

# Não faz parte da sequência formal de "número do envio" (não usa
# registrar_tentativa_envio/MAX_TENTATIVAS_POR_ENVIO): tem seu próprio
# controle de sucesso/erro dentro das abas "Análise" e "Envios", como
# pedido. Uma falha aqui não afeta a sequência normal do dia.
#
# Reaproveita a integração já existente no notebook: conexão com o
# Google Sheets (mesmo padrão de credenciais), o cliente do Gemini, e o
# envio via API do ZapperHub (mesma url/headers/id_grupo_envio já
# definidos lá no topo do notebook, respeitando ENVIAR_PARA_GRUPO_TESTE).

import hashlib
import difflib
import datetime
import xml.etree.ElementTree as ET
from google import genai
from google.genai import types

# ── Meses por extenso em português, pra reconhecer datas tipo "25 de agosto" ──
MESES_PT_NUMERO = {
    "janeiro": 1, "fevereiro": 2, "março": 3, "marco": 3, "abril": 4,
    "maio": 5, "junho": 6, "julho": 7, "agosto": 8, "setembro": 9,
    "outubro": 10, "novembro": 11, "dezembro": 12,
}

def extrair_data_validade(texto, data_referencia):
    """Tenta extrair uma data de validade/expiração de um texto livre
    (título + resumo de uma promoção), cobrindo os formatos mais comuns
    observados nos 10 sites (ex: "23/08/2026 às 10:58", "válida até
    quinta-feira, 27 de agosto", "somente hoje (24)", "até o dia 30 de
    agosto", "Hoje é último dia..."). É um extrator BEST-EFFORT: cobre os
    padrões conhecidos, mas não tem garantia de pegar 100% dos casos.
    Retorna "DD/MM/AAAA" ou "" se não encontrar nada reconhecível.
    """
    if not texto:
        return ""

    texto_norm = texto.lower()

    def resolve_ano_futuro(dia, mes, ano):
        """Se a data cair no passado em relação a hoje, assume que é do
        próximo ano/mês (uma promoção não pode vencer no passado)."""
        try:
            data_candidata = datetime.date(ano, mes, dia)
        except ValueError:
            return None
        if data_candidata < data_referencia:
            try:
                if mes == 12:
                    data_candidata = datetime.date(ano + 1, 1, dia)
                else:
                    data_candidata = datetime.date(ano, mes + 1, dia)
            except ValueError:
                return None
        return data_candidata

    # 1) Data completa DD/MM/AAAA (com ou sem hora junto, ex: "23/08/2026 às 10:58")
    m = re.search(r'\b(\d{1,2})/(\d{1,2})/(\d{4})\b', texto)
    if m:
        dia, mes, ano = int(m.group(1)), int(m.group(2)), int(m.group(3))
        try:
            return datetime.date(ano, mes, dia).strftime("%d/%m/%Y")
        except ValueError:
            pass

    # 2) "DD de MÊS [de AAAA]" por extenso (ex: "25 de agosto de 2026", "27 de agosto")
    m = re.search(
        r'\b(\d{1,2})\s+de\s+(janeiro|fevereiro|março|marco|abril|maio|junho|julho|agosto|setembro|outubro|novembro|dezembro)(?:\s+de\s+(\d{4}))?\b',
        texto_norm
    )
    if m:
        dia = int(m.group(1))
        mes = MESES_PT_NUMERO[m.group(2)]
        if m.group(3):
            try:
                data_candidata = datetime.date(int(m.group(3)), mes, dia)
                return data_candidata.strftime("%d/%m/%Y")
            except ValueError:
                pass
        else:
            data_candidata = resolve_ano_futuro(dia, mes, data_referencia.year)
            if data_candidata:
                return data_candidata.strftime("%d/%m/%Y")

    # 3) "(DD/MM)" entre parênteses, sem ano (ex: "válida até quarta-feira (26/08)")
    m = re.search(r'\((\d{1,2})/(\d{1,2})\)', texto)
    if m:
        dia, mes = int(m.group(1)), int(m.group(2))
        data_candidata = resolve_ano_futuro(dia, mes, data_referencia.year)
        if data_candidata:
            return data_candidata.strftime("%d/%m/%Y")

    # 4) Só o dia entre parênteses, com uma palavra de prazo por perto
    # (ex: "somente hoje (24)", "da segunda-feira (31)") — a checagem de
    # contexto evita confundir com outro número entre parênteses que não
    # tenha nada a ver com data.
    contexto_prazo = r'(hoje|amanh[ãa]|segunda|ter[çc]a|quarta|quinta|sexta|s[áa]bado|domingo|v[áa]lid|at[ée]|termina|encerra)'
    m = re.search(contexto_prazo + r'.{0,40}?\((\d{1,2})\)', texto_norm)
    if m:
        dia = int(m.group(2))
        data_candidata = resolve_ano_futuro(dia, data_referencia.month, data_referencia.year)
        if data_candidata:
            return data_candidata.strftime("%d/%m/%Y")

    # 5) "hoje é (o) último dia" — sem nenhum número, resolve pra hoje
    if re.search(r'hoje\s+[ée]\s+(o\s+)?[úu]ltimo\s+dia', texto_norm):
        return data_referencia.strftime("%d/%m/%Y")

    # 6) "válido/válida até hoje", "somente hoje", "apenas hoje" — sem número junto
    if re.search(r'(v[áa]lid[oa]\s+at[ée]\s+hoje|somente\s+hoje|apenas\s+hoje)', texto_norm):
        return data_referencia.strftime("%d/%m/%Y")

    # 7) "amanhã" sozinho, sem número junto — resolve pra amanhã
    if 'amanh' in texto_norm:
        return (data_referencia + datetime.timedelta(days=1)).strftime("%d/%m/%Y")

    return ""

# ── Cliente do Gemini (instância própria: esta célula pode rodar numa
# execução em que a célula de Transferências Bonificadas, envio 6, não
# roda — então não dá pra depender do `client` definido lá) ──
client_ia_geral = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# ── Conecta na MESMA planilha "Controle de Envios Grupo" já usada pelo
# resto do notebook (mesmo SHEET_ID_CONTROLE_ENVIOS e credenciais), só
# que aqui precisamos do Spreadsheet inteiro (não de uma aba fixa como
# planilha_controle), para acessar as abas "Análise" e "Envios". ──
def conectar_spreadsheet_controle():
    scope = [
        "https://spreadsheets.google.com/feeds",
        "https://www.googleapis.com/auth/drive"
    ]
    creds_dict = json.loads(os.environ["GOOGLE_CREDENTIALS"])
    creds = ServiceAccountCredentials.from_json_keyfile_dict(creds_dict, scope)
    client_gs = gspread.authorize(creds)
    return client_gs.open_by_key(SHEET_ID_CONTROLE_ENVIOS)

spreadsheet_controle = conectar_spreadsheet_controle()
aba_analise = spreadsheet_controle.worksheet("Análise")
aba_envios = spreadsheet_controle.worksheet("Envios")
aba_recusados = spreadsheet_controle.worksheet("Recusados")

# ── Número pessoal do usuário (resumo da execução vai pra cá, não pro
# grupo) — tenta a env var (produção/GitHub Actions) e cai pro arquivo
# local numero_pessoal.txt (mesmo padrão de sheet_id.txt, api_key.txt
# etc já usados no "Modo de Teste no Colab"). ──
def obter_numero_pessoal():
    numero = os.environ.get("NUMERO_PESSOAL")
    if numero:
        return numero.strip()
    if os.path.exists("numero_pessoal.txt"):
        with open("numero_pessoal.txt", "r") as f:
            return f.read().strip()
    return None

# ── Gera um id determinístico a partir do link (mesmo link = mesmo id
# sempre, útil pra deduplicação e rastreabilidade) ──
def gerar_id_promocao(link):
    return hashlib.sha256(link.encode("utf-8")).hexdigest()[:16]

# ── Compara dois títulos e diz se são "muito parecidos" (fallback de
# deduplicação quando o link mudou mas é a mesma promoção) ──
def titulos_parecidos(t1, t2, limite=0.85):
    if not t1 or not t2:
        return False
    norm = lambda t: t.lower().strip()
    return difflib.SequenceMatcher(None, norm(t1), norm(t2)).ratio() >= limite

# ── IA de julgamento (sugestão de aprovar/recusar) — ver nota de manutenção
# no topo do notebook. Por enquanto é só SUGESTÃO: grava em sugestao_ia +
# motivo_ia na aba Análise, mas quem decide (coluna "status") continua
# sendo o usuário manualmente. ──
def obter_titulos_recentes(aba, coluna_nome, limite=15, coluna_fallback=None):
    """Pega os últimos `limite` títulos de uma aba (Envios ou Recusados),
    mais recentes primeiro, pra servir de exemplo (few-shot) no julgamento
    da IA. Devolve lista vazia se a aba não tiver linhas ainda (ex:
    "Recusados" no início, antes de qualquer recusa)."""
    try:
        valores = aba.get_all_values()
    except Exception:
        return []
    if len(valores) <= 1:
        return []
    cabecalho = valores[0]
    idx = idx_coluna(cabecalho, coluna_nome)
    idx_fallback = idx_coluna(cabecalho, coluna_fallback) if coluna_fallback else None
    titulos = []
    for row in valores[1:]:
        titulo = row[idx].strip() if idx is not None and idx < len(row) else ""
        if not titulo and idx_fallback is not None and idx_fallback < len(row):
            titulo = row[idx_fallback].strip()
        if titulo:
            titulos.append(titulo)
    return titulos[-limite:][::-1]


def avaliar_promocao_com_ia(titulo, resumo, data_referencia, precisa_data,
                             exemplos_enviados, exemplos_recusados):
    """Pede pro Gemini uma SUGESTÃO de aprovar/recusar essa promoção, usando
    como referência uma amostra recente do que já foi enviado (aprovado) e
    do que já foi recusado. Se precisa_data=True (a extração por regex não
    achou nada em extrair_data_validade), também tenta uma data de validade
    via IA como complemento.

    Nunca levanta exceção — em caso de erro, devolve sugestão/motivo vazios
    e a promoção segue entrando normalmente na fila, só sem sugestão da IA.
    """
    bloco_enviados = "\n".join(f"- {t}" for t in exemplos_enviados) or "(nenhum exemplo ainda)"
    bloco_recusados = "\n".join(f"- {t}" for t in exemplos_recusados) or "(nenhum exemplo ainda)"

    instrucao_data = (
        "- \"expira_em\": se o texto mencionar uma data de validade/expiração, "
        "devolva no formato DD/MM/AAAA (considere hoje = "
        f"{data_referencia.strftime('%d/%m/%Y')}). Se não mencionar nenhuma data, devolva \"\"."
    ) if precisa_data else (
        "- \"expira_em\": sempre \"\" (não precisa preencher, já temos a data por outro meio)."
    )

    prompt_avaliacao = f"""
    Você ajuda a decidir se uma promoção de milhas, cartões e viagens deve
    ser aprovada (enviada pro grupo de WhatsApp) ou recusada, com base no
    histórico de decisões anteriores do usuário.

    PROMOÇÕES JÁ ENVIADAS ANTES (exemplos do que costuma ser aprovado):
    {bloco_enviados}

    PROMOÇÕES JÁ RECUSADAS ANTES (exemplos do que costuma ser recusado):
    {bloco_recusados}

    NOVA PROMOÇÃO PARA AVALIAR:
    Título: {titulo}
    Resumo: {resumo}

    Baseado no padrão dos exemplos acima (se ainda não houver exemplos
    suficientes, use bom senso sobre o que é relevante pra um grupo de
    milhas/cartões/viagens), essa promoção deve ser aprovada ou recusada?

    Também diga se essa promoção é uma "COMPRA DE MILHAS COM DESCONTO"
    (o usuário compra milhas/pontos direto de um programa, com desconto ou
    bônus percentual na compra — diferente de transferência bonificada
    entre parceiros, que é outra coisa).

    Responda APENAS com um JSON válido, exatamente neste formato:
    {{"sugestao": "Aprovar ou Recusar", "motivo": "frase curta (até 15 palavras) explicando o motivo", "expira_em": "...", "eh_compra_milhas_desconto": true ou false}}

    {instrucao_data}
    """

    try:
        response_avaliacao = client_ia_geral.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=prompt_avaliacao,
            config=types.GenerateContentConfig(
                system_instruction="Você avalia promoções de milhas/cartões/viagens para aprovação, com base no histórico do usuário. Responda sempre em JSON válido.",
                temperature=0.2,
                response_mime_type="application/json",
            ),
        )
        resultado_avaliacao = json.loads(response_avaliacao.text)
        sugestao = resultado_avaliacao.get("sugestao", "").strip()
        motivo = resultado_avaliacao.get("motivo", "").strip()
        expira_em_ia = resultado_avaliacao.get("expira_em", "").strip() if precisa_data else ""
        eh_compra_milhas_desconto = bool(resultado_avaliacao.get("eh_compra_milhas_desconto", False))
        if sugestao not in ("Aprovar", "Recusar"):
            sugestao = ""
        return {
            "sugestao": sugestao,
            "motivo": motivo,
            "expira_em_ia": expira_em_ia,
            "eh_compra_milhas_desconto": eh_compra_milhas_desconto,
        }
    except Exception as e:
        return {
            "sugestao": "", "motivo": f"Erro ao consultar IA: {e}",
            "expira_em_ia": "", "eh_compra_milhas_desconto": False,
        }

# =========================================================================
# 🔎 EXTRAÇÃO — RSS (preferido, quando disponível e liberado no robots.txt)
# =========================================================================
def extrair_via_rss(feed_url, max_itens=5):
    """Lê um feed RSS padrão do WordPress e devolve até max_itens
    promoções (título, resumo, link) — sem chamar IA nenhuma aqui."""
    try:
        resp = requests.get(feed_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
        resp.raise_for_status()
        root = ET.fromstring(resp.content)
    except Exception as e:
        print(f"⚠️ Falha ao ler feed {feed_url}: {e}")
        return []

    itens = []
    for item in root.findall(".//item")[:max_itens]:
        titulo = (item.findtext("title") or "").strip()
        link = (item.findtext("link") or "").strip()
        resumo_html = item.findtext("description") or ""
        # A descrição do RSS geralmente vem com marcação HTML — limpa pra
        # ficar só o texto puro, do mesmo jeito que fazemos com o HTML
        # das páginas de artigo em outras partes do notebook.
        resumo = BeautifulSoup(resumo_html, "html.parser").get_text(strip=True)
        if titulo and link:
            # O RSS não traz a imagem de capa — busca na página do artigo
            # (1 requisição extra por item, só pra pegar o og:image).
            imagem_url = extrair_og_image(link)
            itens.append({"titulo": titulo, "resumo": resumo, "link": link, "expira_em": "", "imagem_url": imagem_url})
    return itens

def extrair_og_image(url_artigo):
    """Busca só o og:image de uma página de artigo — usado pra completar a
    imagem quando a fonte é RSS (o feed em si não traz imagem)."""
    try:
        resp = requests.get(url_artigo, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        tag = soup.find("meta", attrs={"property": "og:image"})
        return tag.get("content", "").strip() if tag else ""
    except Exception as e:
        print(f"⚠️ Falha ao buscar imagem de {url_artigo}: {e}")
        return ""

# =========================================================================
# 🔎 EXTRAÇÃO — HTML (fallback, quando não tem RSS ou o robots.txt bloqueia)
# =========================================================================
def extrair_links_html(url_listagem, filtro_link):
    """Busca a página de listagem e devolve os links de artigo que passam
    no filtro_link (uma função href -> bool específica de cada site)."""
    try:
        resp = requests.get(url_listagem, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
    except Exception as e:
        print(f"⚠️ Falha ao acessar {url_listagem}: {e}")
        return []

    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if filtro_link(href):
            links.append(href)
    # dict.fromkeys() preserva a ordem (mais recente primeiro) e remove
    # duplicatas, igual já fazemos na extração de Transferências Bonificadas.
    return list(dict.fromkeys(links))

def extrair_meta_artigo(url_artigo):
    """Busca og:title / og:description / og:image / article:published_time
    da página do artigo — padrão universal do WordPress, funciona nos 4 sites."""
    try:
        resp = requests.get(url_artigo, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
    except Exception as e:
        print(f"⚠️ Falha ao acessar artigo {url_artigo}: {e}")
        return "", "", "", ""

    def meta(prop):
        tag = soup.find("meta", attrs={"property": prop})
        return tag.get("content", "").strip() if tag else ""

    titulo = meta("og:title")
    resumo = meta("og:description")
    imagem_url = meta("og:image")
    data_pub_iso = meta("article:published_time")
    expira_em = ""  # esses sites não expõem data de validade estruturada
    return titulo, resumo, expira_em, imagem_url

def extrair_via_html(url_listagem, filtro_link, max_itens=5):
    """Descobre os links de artigo na página de listagem e busca
    título/resumo/imagem de cada um (até max_itens)."""
    links = extrair_links_html(url_listagem, filtro_link)[:max_itens]
    itens = []
    for link in links:
        titulo, resumo, expira_em, imagem_url = extrair_meta_artigo(link)
        if titulo:
            itens.append({"titulo": titulo, "resumo": resumo, "link": link, "expira_em": expira_em, "imagem_url": imagem_url})
    return itens

# ── Filtros de link específicos de cada site (só usados na via HTML) ──
def filtro_melhorescartoes(href):
    return (
        href.startswith("https://www.melhorescartoes.com.br/")
        and href.endswith(".html")
        and "/page/" not in href
    )

def filtro_melhoresdestinos(href):
    base = "https://www.melhoresdestinos.com.br/"
    if not href.startswith(base):
        return False
    if any(x in href for x in ["/tag/", "/page/", "/autor/", "/app", "wp-content", "wp-json", "/feed"]):
        return False
    caminho = href[len(base):]
    return caminho.startswith("promocao/") or caminho.startswith("milhas/") or caminho.endswith(".html")

# =========================================================================
# 📋 LISTA DAS 10 FONTES (até 5 promoções por fonte por execução)
# =========================================================================
MAX_POR_FONTE = 5

FONTES = [
    {"nome": "Passageiro de Primeira - Promoções", "tipo": "rss",
     "url": "https://passageirodeprimeira.com/categorias/promocoes/feed/"},
    {"nome": "Passageiro de Primeira - Passagens Aéreas", "tipo": "rss",
     "url": "https://passageirodeprimeira.com/categorias/promocoes/passagens-aereas/feed/"},
    {"nome": "Melhores Cartões - Cartões e Milhas", "tipo": "html",
     "url": "https://www.melhorescartoes.com.br/cartoes-milhas", "filtro": filtro_melhorescartoes},
    {"nome": "Melhores Cartões - Passagens e Milhas", "tipo": "html",
     "url": "https://www.melhorescartoes.com.br/passagens-milhas", "filtro": filtro_melhorescartoes},
    {"nome": "Cartões de Crédito - Notícias", "tipo": "rss",
     "url": "https://www.cartoesdecredito.me/noticia/feed/"},
    {"nome": "Cartões de Crédito - Milhas", "tipo": "rss",
     "url": "https://www.cartoesdecredito.me/milhas/feed/"},
    {"nome": "Cartões de Crédito - Classe Executiva", "tipo": "rss",
     "url": "https://www.cartoesdecredito.me/classe-executiva/feed/"},
    {"nome": "Melhores Destinos - Home", "tipo": "html",
     "url": "https://www.melhoresdestinos.com.br/", "filtro": filtro_melhoresdestinos},
    {"nome": "Melhores Destinos - Hotéis", "tipo": "html",
     "url": "https://www.melhoresdestinos.com.br/hoteis", "filtro": filtro_melhoresdestinos},
    {"nome": "Melhores Destinos - Cartões e Milhas", "tipo": "html",
     "url": "https://www.melhoresdestinos.com.br/cartoes-e-milhas", "filtro": filtro_melhoresdestinos},
]

def coletar_promocoes_gerais():
    """Roda o scraper nas 10 fontes (RSS onde dá, HTML onde não dá) e
    devolve uma lista de dicts brutos (sem passar por IA)."""
    todas = []
    for fonte in FONTES:
        print(f"🔎 Coletando: {fonte['nome']} ({fonte['tipo']})")
        if fonte["tipo"] == "rss":
            itens = extrair_via_rss(fonte["url"], max_itens=MAX_POR_FONTE)
        else:
            itens = extrair_via_html(fonte["url"], fonte["filtro"], max_itens=MAX_POR_FONTE)
        for item in itens:
            item["fonte"] = fonte["nome"]
        print(f"   → {len(itens)} promoções coletadas")
        todas.extend(itens)
    return todas

# =========================================================================
# 🗒️ PASSO 1 — PROCESSAR A ABA "ANÁLISE" (execuções anteriores)
# =========================================================================
# status vazio -> não faz nada (aguardando decisão do usuário)
# status "Recusado" -> não faz nada (fica na aba pra sempre)
# status "Excluir" -> apaga a linha
# status "Aprovado" -> vira candidata a envio no Passo 2 (não envia aqui ainda)
cabecalho_analise = aba_analise.row_values(1)
linhas_analise = aba_analise.get_all_values()[1:]  # pula o cabeçalho

# Índice das colunas pelo cabeçalho real da planilha, pra não depender de
# posição fixa caso a ordem das colunas mude.
def idx_coluna(cabecalho, nome):
    return cabecalho.index(nome) if nome in cabecalho else None

COL_ID = idx_coluna(cabecalho_analise, "id")
COL_DATA_CAPTURA = idx_coluna(cabecalho_analise, "data_captura")
COL_FONTE = idx_coluna(cabecalho_analise, "fonte")
COL_TITULO = idx_coluna(cabecalho_analise, "titulo")
COL_RESUMO = idx_coluna(cabecalho_analise, "resumo")
COL_LINK = idx_coluna(cabecalho_analise, "link")
COL_EXPIRA_EM = idx_coluna(cabecalho_analise, "expira_em")
COL_STATUS = idx_coluna(cabecalho_analise, "status")
COL_IMAGEM_URL = idx_coluna(cabecalho_analise, "imagem_url")

def valor(row, col_idx):
    return row[col_idx].strip() if col_idx is not None and col_idx < len(row) else ""

linhas_para_apagar = []    # números de linha na planilha (Excluir + Recusado, apagadas juntas)
fila_aprovados = []        # [{"linha_num": N, "dados": {...}}]
qtd_recusadas_arquivadas = 0

for i, row in enumerate(linhas_analise):
    linha_num = i + 2  # +1 pelo cabeçalho, +1 porque enumerate começa em 0
    status = valor(row, COL_STATUS)

    if status == "Excluir":
        linhas_para_apagar.append(linha_num)
    elif status == "Recusado":
        # Arquiva na aba "Recusados" antes de apagar da Análise, pra manter
        # o histórico (não fica mais esquecida pra sempre na Análise) e
        # servir de exemplo (few-shot) pra IA julgar futuras promoções.
        aba_recusados.append_row([
            valor(row, COL_ID),
            valor(row, COL_DATA_CAPTURA),
            valor(row, COL_FONTE),
            valor(row, COL_TITULO),
            valor(row, COL_RESUMO),
            valor(row, COL_LINK),
            valor(row, COL_EXPIRA_EM),
            "Recusado",
            valor(row, COL_IMAGEM_URL),
            agora_brasilia.strftime("%d/%m/%Y %H:%M:%S"),
        ], value_input_option="USER_ENTERED")
        linhas_para_apagar.append(linha_num)
        qtd_recusadas_arquivadas += 1
    elif status == "Aprovado":
        fila_aprovados.append({
            "linha_num": linha_num,
            "dados": {
                "id": valor(row, COL_ID),
                "data_captura": valor(row, COL_DATA_CAPTURA),
                "fonte": valor(row, COL_FONTE),
                "titulo": valor(row, COL_TITULO),
                "resumo": valor(row, COL_RESUMO),
                "link": valor(row, COL_LINK),
                "expira_em": valor(row, COL_EXPIRA_EM),
                "imagem_url": valor(row, COL_IMAGEM_URL),
            }
        })
    # vazio -> não faz nada, aguardando decisão do usuário

# Apaga de baixo pra cima (Excluir + Recusado juntas numa única passada),
# senão os números de linha das próximas exclusões ficam errados a cada
# delete_rows.
for linha_num in sorted(linhas_para_apagar, reverse=True):
    aba_analise.delete_rows(linha_num)
if linhas_para_apagar:
    print(f"🗑️ {len(linhas_para_apagar)} linha(s) removida(s) da aba Análise "
          f"({qtd_recusadas_arquivadas} arquivada(s) em 'Recusados', "
          f"{len(linhas_para_apagar) - qtd_recusadas_arquivadas} excluída(s) direto).")

print(f"📥 {len(fila_aprovados)} promoção(ões) com status 'Aprovado' encontrada(s) na aba Análise.")

# =========================================================================
# 🤖 PASSO 2 — ENVIAR PROMOÇÕES APROVADAS (com IA), NO MÁXIMO 1 POR EXECUÇÃO
# =========================================================================
# Se houver mais de 1 aprovada, as excedentes ficam pendentes pra próxima
# execução — isso é esperado, não é erro.
MAX_ENVIOS_APROVADOS_POR_EXECUCAO = 1
fila_desta_execucao = fila_aprovados[:MAX_ENVIOS_APROVADOS_POR_EXECUCAO]
pendentes_por_limite = max(0, len(fila_aprovados) - MAX_ENVIOS_APROVADOS_POR_EXECUCAO)

qtd_enviadas = 0
qtd_erros = 0

# Linhas da aba Análise já foram deslocadas pelas exclusões do Passo 1,
# mas as exclusões (status "Excluir") e as aprovações (status "Aprovado")
# nunca coexistem na mesma execução como o mesmo número de linha original
# de forma conflitante, já que calculamos linhas_para_excluir e
# fila_aprovados a partir do MESMO snapshot antes de deletar nada. Ainda
# assim, como já apagamos as linhas "Excluir" acima, os números de linha
# dos itens em fila_aprovados podem ter mudado se havia uma linha
# "Excluir" ACIMA deles na planilha. Por segurança, recarregamos os
# números de linha atuais buscando pelo "id" antes de mexer em cada uma.
def localizar_linha_por_id(aba, coluna_id_letra_idx, id_procurado):
    """Recarrega a aba e devolve o número da linha (1-indexado) que tem
    esse id na coluna 'id' — usado pra não mexer na linha errada caso os
    números tenham mudado por causa de deleções anteriores."""
    valores = aba.get_all_values()
    for i, row in enumerate(valores):
        if i == 0:
            continue  # cabeçalho
        if coluna_id_letra_idx < len(row) and row[coluna_id_letra_idx].strip() == id_procurado:
            return i + 1
    return None

for indice_item, item in enumerate(fila_desta_execucao):
    dados = item["dados"]
    titulo_bruto = dados["titulo"]
    resumo_bruto = dados["resumo"]
    link = dados["link"]

    print(f"\n{'='*50}")
    print(f"🤖 Formatando com IA: {titulo_bruto[:60]}...")

    # ── Prompt seguindo o guia de voz de marca do projeto ──
    prompt_formatacao = f"""
    Você é o redator do grupo de WhatsApp sobre milhas, cartões e viagens.
    Vai transformar uma promoção bruta (extraída de um site) em uma
    mensagem curta e chamativa pra mandar no grupo.

    REGRAS DE VOZ DA MARCA (siga à risca):
    - Se dirija ao público como "você" ou "galera" — mas VARIE a abertura
      da mensagem a cada promoção, não comece toda mensagem com a mesma
      palavra. Alterne entre estilos diferentes de abertura, por exemplo:
      "Galera, ...", "Você que curte milhas vai gostar dessa...", "Saiu
      uma boa...", "Se liga nessa...", "Olha que oportunidade...", ou
      comece direto pelo fato/número mais chamativo, sem vocativo nenhum.
      Escolha um estilo diferente a cada vez, não repita sempre o mesmo.
    - Antes de qualquer informação negativa (ex: promoção só até tal data,
      vagas limitadas, restrição), use a palavra "infelizmente".
    - Sempre que possível, use números concretos (valores, percentuais,
      quantidade de pontos/milhas) em vez de termos vagos.
    - NUNCA use travessão (—) nem em dash. Use vírgula, ponto ou parênteses.
    - Seja direto e chamativo, sem inventar informação que não esteja no
      texto original.

    ÍCONE DO TÍTULO:
    - Escolha 1 emoji que faça sentido pro assunto real da promoção (ex:
      ✈️ passagem aérea, 💳 cartão de crédito, 🏨 hotel/hospedagem, 💰
      cashback/pontos, 🎫 passeio/ingresso, 🔥 oferta muito boa/urgente).
      Baseie a escolha no conteúdo de verdade, não escolha aleatoriamente.

    TÍTULO BRUTO DA FONTE:
    {titulo_bruto}

    RESUMO BRUTO DA FONTE:
    {resumo_bruto}

    Gere:
    - "icone": um único emoji que combine com o assunto da promoção.
    - "titulo": chamativo, curto (1 linha), SEM o emoji dentro do texto
      (o emoji é um campo separado).
    - "resumo": curto (2 a 4 frases), seguindo as regras de voz acima
      (abertura variada, "infelizmente" antes de negativas, números
      concretos, sem travessão).

    Responda APENAS com um JSON válido, sem nenhum texto antes ou depois,
    exatamente neste formato:
    {{"icone": "...", "titulo": "...", "resumo": "..."}}
    """

    icone_ia = ""
    titulo_ia = ""
    resumo_ia = ""
    erro_ia = None
    try:
        # response_mime_type="application/json" força o Gemini a devolver
        # JSON válido (em vez de texto livre) — evita a fragilidade de
        # tentar extrair título/resumo com regex de uma mensagem formatada
        # em texto, como fazíamos antes na extração de Transferências
        # Bonificadas.
        response_ia = client_ia_geral.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=prompt_formatacao,
            config=types.GenerateContentConfig(
                system_instruction="Você formata promoções de milhas para WhatsApp seguindo a voz da marca. Responda sempre em JSON válido.",
                temperature=0.3,
                response_mime_type="application/json",
            ),
        )
        resultado_ia = json.loads(response_ia.text)
        icone_ia = resultado_ia.get("icone", "").strip()
        titulo_ia = resultado_ia.get("titulo", "").strip()
        resumo_ia = resultado_ia.get("resumo", "").strip()
        if not titulo_ia or not resumo_ia:
            erro_ia = "IA retornou título ou resumo vazio"
    except Exception as e:
        erro_ia = f"Erro ao formatar com IA: {e}"

    if erro_ia:
        print(f"❌ {erro_ia}")
        qtd_erros += 1
        # Não mexe na linha (continua "Aprovado" na Análise, elegível
        # pra nova tentativa na próxima execução) — nunca desaparece.
        continue

    # ── Monta o título final: ícone (se a IA devolveu um) + negrito no
    # padrão do WhatsApp (*texto* — um asterisco só, não dois como no
    # Markdown normal, senão não fica em negrito no WhatsApp de verdade). ──
    titulo_formatado = f"{icone_ia} *{titulo_ia}*".strip() if icone_ia else f"*{titulo_ia}*"

    mensagem_final = f"{titulo_formatado}\n\n{resumo_ia}\n\n🔗 {link}"
    imagem_url = dados.get("imagem_url", "")

    # ── Envia pro GRUPO via ZapperHub (mesma integração já existente:
    # url/url_media, headers e id_grupo_envio já definidos no topo do
    # notebook, respeitando a flag ENVIAR_PARA_GRUPO_TESTE). Se a promoção
    # tem imagem de capa, manda como mídia (mesmo padrão dos Jabares); se
    # não tem (ou a extração falhou), cai pro envio só de texto. ──
    if imagem_url:
        payload_envio = {
            "jid": id_grupo_envio,
            "mediaType": "image",
            "mimetype": "image/jpeg",
            "media": imagem_url,
            "caption": mensagem_final,
            "filename": "promocao.jpg",
        }
        endpoint_envio = url_media
    else:
        payload_envio = {
            "jid": id_grupo_envio,
            "message": mensagem_final,
        }
        endpoint_envio = url

    try:
        response_envio = requests.post(endpoint_envio, json=payload_envio, headers=headers, timeout=30)
        sucesso_envio = response_envio.status_code == 200
        erro_envio = None if sucesso_envio else f"{response_envio.status_code} - {response_envio.text}"
    except Exception as e:
        sucesso_envio = False
        erro_envio = f"Exceção ao enviar: {e}"

    if sucesso_envio:
        print("📨 Promoção enviada com sucesso para o grupo!")
        qtd_enviadas += 1

        agora_str = agora_brasilia.strftime("%d/%m/%Y %H:%M:%S")
        linha_envios = [
            dados["id"],
            dados["data_captura"],
            dados["fonte"],
            titulo_formatado,
            resumo_ia,
            link,
            agora_str,
            "enviado",
            "",
            titulo_bruto,  # título original da fonte (antes da IA reescrever) —
                            # usado pro Passo 3 comparar título bruto com título
                            # bruto (mais preciso que comparar com o formatado)
        ]
        aba_envios.append_row(linha_envios, value_input_option="USER_ENTERED")

        # Remove da Análise só depois de confirmar que gravou em Envios,
        # pra linha nunca "desaparecer sem rastro" em caso de falha entre
        # os dois passos.
        linha_atual = localizar_linha_por_id(aba_analise, COL_ID, dados["id"])
        if linha_atual:
            aba_analise.delete_rows(linha_atual)
    else:
        print(f"❌ Erro ao enviar: {erro_envio}")
        qtd_erros += 1
        # Não remove da Análise — continua "Aprovado", elegível pra nova
        # tentativa na próxima execução (nunca desaparece sem rastro).

    # Aguarda no mínimo 1 minuto antes do próximo envio da fila.
    if indice_item < len(fila_desta_execucao) - 1:
        time.sleep(60)


if numero_envio_hoje in (1, 4):
  # =========================================================================
  # 🆕 PASSO 3 — CAPTURAR PROMOÇÕES NOVAS (sem IA nesta etapa)
  # =========================================================================
  print(f"\n{'='*50}")
  print("🔎 Iniciando coleta de promoções novas nas 10 fontes...")
  promocoes_coletadas = coletar_promocoes_gerais()

  # ── Carrega o que já existe (Análise + Envios) pra não duplicar ──
  # Recarrega a Análise (pode ter mudado nos Passos 1 e 2).
  linhas_analise_atual = aba_analise.get_all_values()[1:]
  linhas_envios_atual = aba_envios.get_all_values()[1:]
  cabecalho_envios = aba_envios.row_values(1)
  COL_ENVIOS_LINK = idx_coluna(cabecalho_envios, "link")
  COL_ENVIOS_TITULO_FORMATADO = idx_coluna(cabecalho_envios, "titulo_formatado")
  COL_ENVIOS_TITULO_BRUTO = idx_coluna(cabecalho_envios, "titulo_bruto")

  links_existentes = set()
  titulos_existentes = []

  for row in linhas_analise_atual:
      links_existentes.add(valor(row, COL_LINK))
      titulos_existentes.append(valor(row, COL_TITULO))

  for row in linhas_envios_atual:
      links_existentes.add(valor(row, COL_ENVIOS_LINK))
      # Prefere o título bruto (comparação "maçã com maçã" contra o que
      # acabamos de raspar, que também é bruto) — cai pro título formatado
      # pela IA só em linhas antigas gravadas antes dessa coluna existir.
      titulo_bruto_envio = valor(row, COL_ENVIOS_TITULO_BRUTO)
      titulos_existentes.append(titulo_bruto_envio or valor(row, COL_ENVIOS_TITULO_FORMATADO))

  qtd_novas = 0
  agora_str_captura = agora_brasilia.strftime("%d/%m/%Y %H:%M:%S")

  # Amostra recente de Enviados/Recusados, buscada uma única vez (não a
  # cada promoção) pra servir de referência (few-shot) no julgamento da IA.
  exemplos_enviados = obter_titulos_recentes(
      aba_envios, "titulo_bruto", limite=15, coluna_fallback="titulo_formatado"
  )
  exemplos_recusados = obter_titulos_recentes(aba_recusados, "titulo", limite=15)

  for promo in promocoes_coletadas:
      link = promo["link"]
      titulo = promo["titulo"]

      if link in links_existentes:
          continue  # já existe por link exato

      duplicada_por_titulo = any(titulos_parecidos(titulo, t) for t in titulos_existentes)
      if duplicada_por_titulo:
          continue  # já existe por título muito parecido

      novo_id = gerar_id_promocao(link)

      # Tenta extrair a data de validade do próprio título/resumo (best
      # effort — ver extrair_data_validade lá em cima). Se a fonte já
      # tinha mandado algo em "expira_em" (nenhuma manda hoje, mas por
      # segurança), usa isso primeiro.
      expira_em = promo.get("expira_em", "") or extrair_data_validade(
          f"{titulo} {promo['resumo']}", agora_brasilia.date()
      )

      # SUGESTÃO da IA (aprovar/recusar) — ainda não decide sozinha, só
      # grava a sugestão + motivo pro usuário conferir. Se o regex acima
      # não achou data de validade, pede pra IA tentar também.
      avaliacao_ia = avaliar_promocao_com_ia(
          titulo, promo["resumo"], agora_brasilia.date(),
          precisa_data=not bool(expira_em),
          exemplos_enviados=exemplos_enviados,
          exemplos_recusados=exemplos_recusados,
      )
      if not expira_em and avaliacao_ia["expira_em_ia"]:
          expira_em = avaliacao_ia["expira_em_ia"]

      # 🔔 Alerta pessoal: promoção de compra de milhas com desconto
      if avaliacao_ia["eh_compra_milhas_desconto"]:
          enviar_alerta_pessoal(
              "Compra de milhas com desconto",
              f"{titulo}\n{link}"
          )

      nova_linha = [
          novo_id,
          agora_str_captura,
          promo["fonte"],
          titulo,
          promo["resumo"],
          link,
          expira_em,
          "",  # status vazio por padrão — aguardando decisão do usuário
          promo.get("imagem_url", ""),
          avaliacao_ia["sugestao"],
          avaliacao_ia["motivo"],
      ]
      aba_analise.append_row(nova_linha, value_input_option="USER_ENTERED")

      # Atualiza as listas locais pra não duplicar entre itens desta mesma
      # coleta (ex: a mesma promoção aparecendo em 2 fontes diferentes).
      links_existentes.add(link)
      titulos_existentes.append(titulo)
      qtd_novas += 1

  print(f"🆕 {qtd_novas} promoção(ões) nova(s) adicionada(s) à aba Análise (de {len(promocoes_coletadas)} coletadas).")

  # ── Reordena a aba Análise pela data de validade (mais próxima de
  # vencer primeiro), pra promoções mais urgentes ficarem no topo, prontas
  # pra serem revisadas/aprovadas antes de vencer. Linhas sem expira_em
  # (em branco, ou que o extrator não conseguiu reconhecer) ficam por
  # último. ──
  def reordenar_analise_por_expiracao(aba):
      cabecalho_atual = aba.row_values(1)
      col_expira_atual = idx_coluna(cabecalho_atual, "expira_em")
      if col_expira_atual is None:
          return

      dados_atuais = aba.get_all_values()
      linhas_atuais = dados_atuais[1:]
      if len(linhas_atuais) < 2:
          return  # nada pra reordenar

      def chave_ordenacao(linha):
          valor_expira = linha[col_expira_atual].strip() if col_expira_atual < len(linha) else ""
          try:
              return (0, datetime.datetime.strptime(valor_expira, "%d/%m/%Y").date())
          except ValueError:
              return (1, datetime.date.max)  # sem data (ou inválida) vai pro final

      linhas_ordenadas = sorted(linhas_atuais, key=chave_ordenacao)

      if linhas_ordenadas != linhas_atuais:
          aba.update("A2", linhas_ordenadas, value_input_option="USER_ENTERED")
          print("↕️ Aba Análise reordenada por data de expiração (mais urgente primeiro).")

  reordenar_analise_por_expiracao(aba_analise)
else:
  print("⏭️ Fora da janela de captura (só roda nos envios 1 e 4) — pulando coleta de promoções novas nesta execução.")
  qtd_novas = 0
  promocoes_coletadas = []

# =========================================================================
# 📊 PASSO 4 — NOTIFICAÇÃO DE RESUMO (pro número pessoal, não pro grupo)
# =========================================================================
numero_pessoal = obter_numero_pessoal()

# Texto do "envio X de 7" — trata o caso em que numero_envio_hoje já
# passou de 7 (todos os envios normais do dia já foram resolvidos antes
# desta execução chegar aqui), pra não mostrar algo tipo "Envio 8 de 7".
if numero_envio_hoje <= len(NOME_BLOCO_POR_NUMERO):
    texto_envio_do_dia = f"🔢 Envio {numero_envio_hoje} de {len(NOME_BLOCO_POR_NUMERO)} hoje"
else:
    texto_envio_do_dia = f"🔢 Todos os {len(NOME_BLOCO_POR_NUMERO)} envios de hoje já foram feitos"

# Monta o link a partir do secret SHEET_ID_CONTROLE_ENVIOS (já usado pra
# conectar na planilha lá em cima) — assim nunca fica desatualizado se a
# planilha for trocada e o secret for atualizado.
LINK_PLANILHA_CONTROLE = f"https://docs.google.com/spreadsheets/d/{SHEET_ID_CONTROLE_ENVIOS}/edit"

resumo_execucao = textwrap.dedent(f"""
Resumo da execução:
{texto_envio_do_dia}
✅ {qtd_enviadas} promoções enviadas ao grupo
🆕 {qtd_novas} novas promoções capturadas, aguardando aprovação
⚠️ {qtd_erros} erros de envio
📋 {pendentes_por_limite} promoções pendentes na fila (aprovadas mas não enviadas por limite de {MAX_ENVIOS_APROVADOS_POR_EXECUCAO})

🔗 Planilha: {LINK_PLANILHA_CONTROLE}
""").strip()

print(f"\n{'='*50}")
print(resumo_execucao)

if numero_pessoal:
    # Sem supressao por "nada relevante" - o usuario quer receber esse
    # resumo (com o numero do envio e o link da planilha) em toda execucao,
    # independente de ter enviado/capturado/errado algo ou nao.
    payload_resumo = {
        "jid": numero_pessoal,
        "message": resumo_execucao,
    }
    try:
        response_resumo = requests.post(url, json=payload_resumo, headers=headers, timeout=30)
        if response_resumo.status_code == 200:
            print("✅ Resumo enviado para o número pessoal!")
        else:
            print(f"⚠️ Erro ao enviar resumo: {response_resumo.status_code} - {response_resumo.text}")
    except Exception as e:
        print(f"⚠️ Exceção ao enviar resumo: {e}")
else:
    print("⚠️ NUMERO_PESSOAL não configurado (nem env var, nem numero_pessoal.txt) — resumo não enviado.")

## 🗂️ Extração Livelo e Esfera (parceiros e pontuação) — DESATIVADO

Coleta o catálogo de parceiros e pontuação da Livelo e da Esfera, grava na aba **"Extração Livelo e Esfera"**. Por enquanto essa célula só coleta e guarda o dado — nada no notebook usa isso ainda.

**Está desativada de propósito** (`EXTRACAO_PARCEIROS_ATIVA = False`). Antes de ativar, é preciso:
1. Criar a aba **"Extração Livelo e Esfera"** na planilha real do Google Sheets, com as colunas: `data_extracao, programa_fidelidade, parceiro, pontuacao, pontuacao_clube, moeda, link_parceiro`
2. Adicionar `playwright` no `requirements.txt`
3. Adicionar um passo no workflow do GitHub Actions instalando o navegador (`playwright install --with-deps chromium`) — ver `teste_esfera_playwright.yml` no repositório como exemplo
4. Trocar `EXTRACAO_PARCEIROS_ATIVA` para `True`

A Livelo usa só `requests` (o HTML já vem com os dados prontos, sem precisar de navegador). A Esfera precisa de Playwright (navegador de verdade) porque o site só carrega os parceiros via JavaScript — já testamos isso rodando no GitHub Actions e funcionou (ver `teste_esfera_playwright.py`).

In [ ]:
# =========================================================================
# 🗂️ EXTRAÇÃO LIVELO E ESFERA (parceiros e pontuação) — DESATIVADO
# =========================================================================
# Ver a célula markdown acima para o passo a passo de como ativar.
# As funções extrair_parceiros_livelo()/extrair_parceiros_esfera() ficam
# definidas lá no início do notebook (logo após os imports), porque outras
# categorias (Farmácia, Suplementos, Café, Vinho, Mercado) também usam
# elas e precisam que estejam definidas bem antes no notebook.
EXTRACAO_PARCEIROS_ATIVA = False

if EXTRACAO_PARCEIROS_ATIVA and numero_envio_hoje == 1:
    aba_extracao_parceiros = spreadsheet_controle_envios.worksheet("Extração Livelo e Esfera")
    data_extracao_parceiros = agora_brasilia.strftime("%d/%m/%Y %H:%M:%S")

    try:
        parceiros_livelo = extrair_parceiros_livelo()
        registrar_extracao_parceiros(aba_extracao_parceiros, parceiros_livelo, "Livelo", data_extracao_parceiros)
        print(f"✅ {len(parceiros_livelo)} parceiros da Livelo gravados.")
    except Exception as e:
        print(f"⚠️ Erro ao extrair/gravar parceiros da Livelo: {e}")

    try:
        parceiros_esfera = extrair_parceiros_esfera()
        registrar_extracao_parceiros(aba_extracao_parceiros, parceiros_esfera, "Esfera", data_extracao_parceiros)
        print(f"✅ {len(parceiros_esfera)} parceiros da Esfera gravados.")
    except Exception as e:
        print(f"⚠️ Erro ao extrair/gravar parceiros da Esfera: {e}")
else:
    print("⏭️ Extração Livelo/Esfera desativada (ou não é o envio 1) — pulando.")
